In [0]:
from pyspark.sql.functions import col, trim, expr

# Read Bronze collisions table
collisions = spark.table("bronze_collisions")

# Create Silver Candidate with type conversions and text standardisation
collisions_candidate = (
    collisions

    # -------------------------
    # Identifier columns
    # -------------------------
    .withColumn("collision_index", trim(col("collision_index")))
    .withColumn("collision_ref_no", trim(col("collision_ref_no")))

    # -------------------------
    # Numeric columns
    # -------------------------
    .withColumn("collision_year", expr("try_cast(collision_year AS INT)"))
    .withColumn("location_easting_osgr", expr("try_cast(location_easting_osgr AS INT)"))
    .withColumn("location_northing_osgr", expr("try_cast(location_northing_osgr AS INT)"))
    .withColumn("longitude", expr("try_cast(longitude AS DOUBLE)"))
    .withColumn("latitude", expr("try_cast(latitude AS DOUBLE)"))
    .withColumn("police_force", expr("try_cast(police_force AS INT)"))
    .withColumn("collision_severity", expr("try_cast(collision_severity AS INT)"))
    .withColumn("number_of_vehicles", expr("try_cast(number_of_vehicles AS INT)"))
    .withColumn("number_of_casualties", expr("try_cast(number_of_casualties AS INT)"))
    .withColumn("day_of_week", expr("try_cast(day_of_week AS INT)"))
    .withColumn("local_authority_district", expr("try_cast(local_authority_district AS INT)"))
    .withColumn("first_road_class", expr("try_cast(first_road_class AS INT)"))
    .withColumn("first_road_number", expr("try_cast(first_road_number AS INT)"))
    .withColumn("road_type", expr("try_cast(road_type AS INT)"))
    .withColumn("speed_limit", expr("try_cast(speed_limit AS INT)"))
    .withColumn("junction_detail_historic", expr("try_cast(junction_detail_historic AS INT)"))
    .withColumn("junction_detail", expr("try_cast(junction_detail AS INT)"))
    .withColumn("junction_control", expr("try_cast(junction_control AS INT)"))
    .withColumn("second_road_class", expr("try_cast(second_road_class AS INT)"))
    .withColumn("second_road_number", expr("try_cast(second_road_number AS INT)"))
    .withColumn(
        "pedestrian_crossing_human_control_historic",
        expr("try_cast(pedestrian_crossing_human_control_historic AS INT)")
    )
    .withColumn(
        "pedestrian_crossing_physical_facilities_historic",
        expr("try_cast(pedestrian_crossing_physical_facilities_historic AS INT)")
    )
    .withColumn(
        "pedestrian_crossing",
        expr("try_cast(pedestrian_crossing AS INT)")
    )
    .withColumn("light_conditions", expr("try_cast(light_conditions AS INT)"))
    .withColumn("weather_conditions", expr("try_cast(weather_conditions AS INT)"))
    .withColumn(
        "road_surface_conditions",
        expr("try_cast(road_surface_conditions AS INT)")
    )
    .withColumn(
        "special_conditions_at_site",
        expr("try_cast(special_conditions_at_site AS INT)")
    )
    .withColumn(
        "carriageway_hazards_historic",
        expr("try_cast(carriageway_hazards_historic AS INT)")
    )
    .withColumn(
        "carriageway_hazards",
        expr("try_cast(carriageway_hazards AS INT)")
    )
    .withColumn(
        "urban_or_rural_area",
        expr("try_cast(urban_or_rural_area AS INT)")
    )
    .withColumn(
        "did_police_officer_attend_scene_of_accident",
        expr("try_cast(did_police_officer_attend_scene_of_accident AS INT)")
    )
    .withColumn(
        "trunk_road_flag",
        expr("try_cast(trunk_road_flag AS INT)")
    )
    .withColumn(
        "enhanced_severity_collision",
        expr("try_cast(enhanced_severity_collision AS INT)")
    )
    .withColumn(
        "collision_injury_based",
        expr("try_cast(collision_injury_based AS INT)")
    )
    .withColumn(
        "collision_adjusted_severity_serious",
        expr("try_cast(collision_adjusted_severity_serious AS INT)")
    )
    .withColumn(
        "collision_adjusted_severity_slight",
        expr("try_cast(collision_adjusted_severity_slight AS INT)")
    )

    # -------------------------
    # Text standardisation
    # -------------------------
    .withColumn("date", trim(col("date")))
    .withColumn("time", trim(col("time")))
    .withColumn(
        "local_authority_ons_district",
        trim(col("local_authority_ons_district"))
    )
    .withColumn(
        "local_authority_highway",
        trim(col("local_authority_highway"))
    )
    .withColumn(
        "local_authority_highway_current",
        trim(col("local_authority_highway_current"))
    )
    .withColumn(
        "lsoa_of_accident_location",
        trim(col("lsoa_of_accident_location"))
    )
)

print("Silver Candidate DataFrame created successfully.")

Silver Candidate DataFrame created successfully.


In [0]:
from pyspark.sql.functions import col, concat_ws, expr

collisions_candidate = collisions_candidate.withColumn(
    "collision_timestamp",
    expr("""
        try_to_timestamp(
            concat_ws(' ', date, time),
            'dd/MM/yyyy HH:mm'
        )
    """)
)

print("collision_timestamp created successfully")

collision_timestamp created successfully


In [0]:
display(
    collisions_candidate
    .filter(col("collision_timestamp").isNull())
    .select(
        "collision_index",
        "date",
        "time",
        "collision_timestamp"
    )
)

collision_index,date,time,collision_timestamp
2024430578902,31/02/2024,20:55,null
2024041406607,31/02/2024,20:10,null
2024221431708,22/03/2024,25:61,null
2024010507883,31/02/2024,08:49,null
2024122400699,01/06/2024,25:61,null
2024161456759,31/02/2024,08:35,null
2024302400098,31/02/2024,16:30,null
2024440111873,15/03/2024,25:61,null
2024991450470,31/02/2024,20:55,null
2024010549880,31/02/2024,11:30,null


In [0]:
from pyspark.sql.functions import month, hour, dayofweek

collisions_candidate = (
    collisions_candidate
    .withColumn("collision_month", month("collision_timestamp"))
    .withColumn("collision_hour", hour("collision_timestamp"))
    .withColumn("collision_weekday", dayofweek("collision_timestamp"))
)

In [0]:
from pyspark.sql.functions import when

collisions_candidate = collisions_candidate.withColumn(
    "weekend_flag",
    when(col("collision_weekday").isin(1, 7), 1)
    .otherwise(0)
)

In [0]:
collisions_candidate = collisions_candidate.withColumn(
    "time_band",
    when(col("collision_hour").between(0, 5), "00:00-05:59")
    .when(col("collision_hour").between(6, 11), "06:00-11:59")
    .when(col("collision_hour").between(12, 17), "12:00-17:59")
    .when(col("collision_hour").between(18, 23), "18:00-23:59")
    .otherwise("Unknown")
)

In [0]:
collisions_candidate = collisions_candidate.withColumn(
    "severe_collision_flag",
    when(col("collision_severity").isin(1, 2), 1)
    .otherwise(0)
)

In [0]:
collisions_candidate = collisions_candidate.withColumn(
    "urban_rural_group",
    when(col("urban_or_rural_area") == 1, "Urban")
    .when(col("urban_or_rural_area") == 2, "Rural")
    .otherwise("Unknown")
)

In [0]:
display(
    collisions_candidate.select(
        "collision_index",
        "collision_timestamp",
        "collision_month",
        "collision_hour",
        "collision_weekday",
        "weekend_flag",
        "time_band",
        "collision_severity",
        "severe_collision_flag",
        "urban_or_rural_area",
        "urban_rural_group",
        "ingested_at",
        "source_file"
    ).limit(20)
)

collision_index,collision_timestamp,collision_month,collision_hour,collision_weekday,weekend_flag,time_band,collision_severity,severe_collision_flag,urban_or_rural_area,urban_rural_group,ingested_at,source_file
202417M119024,2024-12-05T16:10:00.000Z,12,16,5,0,12:00-17:59,3,0,1,Urban,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
202417S312124,2024-10-22T14:56:00.000Z,10,14,3,0,12:00-17:59,3,0,1,Urban,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024070110901,2024-01-24T17:50:00.000Z,1,17,4,0,12:00-17:59,3,0,2,Rural,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024041446676,2024-05-22T17:45:00.000Z,5,17,4,0,12:00-17:59,2,1,2,Rural,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024041478641,2024-08-11T15:36:00.000Z,8,15,1,1,12:00-17:59,2,1,1,Urban,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024161425653,2024-03-19T14:40:00.000Z,3,14,3,0,12:00-17:59,3,0,2,Rural,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024161438773,2024-05-03T08:19:00.000Z,5,8,6,0,06:00-11:59,3,0,1,Urban,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024122400772,2024-05-30T08:29:00.000Z,5,8,5,0,06:00-11:59,3,0,1,Urban,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024302400537,2024-04-18T08:25:00.000Z,4,8,5,0,06:00-11:59,3,0,1,Urban,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024302400805,2024-05-24T17:10:00.000Z,5,17,6,0,12:00-17:59,3,0,1,Urban,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv


In [0]:
bronze_count = spark.table("bronze_collisions").count()
candidate_count = collisions_candidate.count()

print("Bronze collisions   :", bronze_count)
print("Silver Candidate    :", candidate_count)
print("Row counts match    :", bronze_count == candidate_count)

Bronze collisions   : 101527
Silver Candidate    : 101527
Row counts match    : True


In [0]:
collisions_candidate.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_collisions_candidate")

print("silver_collisions_candidate created successfully")

silver_collisions_candidate created successfully


In [0]:
display(
    spark.table("silver_collisions_candidate").limit(10)
)

collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,local_authority_district,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail_historic,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control_historic,pedestrian_crossing_physical_facilities_historic,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,ingested_at,source_file,collision_timestamp,collision_month,collision_hour,collision_weekday,weekend_flag,time_band,severe_collision_flag,urban_rural_group
202417M119024,2024,17M119024,450057,519938,-1.22722,54.57219,17,3,2,1,05/12/2024,5,16:10,-1,E06000002,E06000002,E06000002,6,0,6,30,-1,0,-1,-1,-1,-1,-1,0,4,2,2,-1,-1,0,1,3,2,E01035190,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-12-05T16:10:00.000Z,12,16,5,0,12:00-17:59,0,Urban
202417S312124,2024,17S312124,445858,516830,-1.29265,54.54466,17,3,1,1,22/10/2024,3,14:56,-1,E06000004,E06000004,E06000004,6,0,6,30,-1,0,-1,-1,-1,-1,-1,0,1,1,1,-1,-1,0,1,3,2,E01012239,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-10-22T14:56:00.000Z,10,14,3,0,12:00-17:59,0,Urban
2024070110901,2024,070110901,364765,390979,-2.53157,53.41443,7,3,2,1,24/01/2024,4,17:50,-1,E06000007,E06000007,E06000007,6,0,6,30,3,13,4,6,0,0,0,0,4,1,1,0,0,0,2,2,2,E01012459,-1,0,null,null,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-01-24T17:50:00.000Z,1,17,4,0,12:00-17:59,0,Rural
2024041446676,2024,041446676,365989,419804,-2.5163,53.67359,4,2,1,1,22/05/2024,4,17:45,-1,E06000008,E06000008,E06000008,3,675,6,50,-1,0,-1,-1,-1,-1,-1,0,1,5,5,-1,-1,0,2,1,2,E01012628,6,1,1,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-22T17:45:00.000Z,5,17,4,0,12:00-17:59,1,Rural
2024041478641,2024,041478641,366864,430071,-2.50416,53.76592,4,2,2,1,11/08/2024,1,15:36,-1,E06000008,E06000008,E06000008,3,6119,3,50,-1,16,4,6,0,-1,-1,0,1,1,1,-1,-1,0,1,1,2,E01012581,7,1,1,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-08-11T15:36:00.000Z,8,15,1,1,12:00-17:59,1,Urban
2024161425653,2024,161425653,526509,404543,-0.09323,53.52234,16,3,2,2,19/03/2024,3,14:40,-1,E06000012,E06000012,E06000012,4,1203,6,40,-1,0,-1,6,0,-1,-1,0,1,9,1,-1,-1,0,2,2,2,E01013215,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-03-19T14:40:00.000Z,3,14,3,0,12:00-17:59,0,Rural
2024161438773,2024,161438773,488486,409919,-0.66503,53.57841,16,3,1,1,03/05/2024,6,08:19,-1,E06000013,E06000013,E06000013,6,0,6,30,-1,0,-1,6,0,-1,-1,0,1,1,1,-1,-1,0,1,1,2,E01013321,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-03T08:19:00.000Z,5,8,6,0,06:00-11:59,0,Urban
2024122400772,2024,122400772,458988,455182,-1.10184,53.98933,12,3,4,1,30/05/2024,5,08:29,-1,E06000014,E06000014,E06000014,6,0,6,30,0,0,-1,0,-1,0,0,0,1,1,1,0,0,0,1,1,2,E01013431,-1,0,null,null,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-30T08:29:00.000Z,5,8,5,0,06:00-11:59,0,Urban
2024302400537,2024,302400537,438527,333460,-1.42873,52.89721,30,3,2,2,18/04/2024,5,08:25,-1,E06000015,E06000015,E06000015,3,5111,1,30,1,0,4,3,6,0,4,14,1,1,1,0,0,0,1,1,2,E01013473,-1,0,null,null,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-04-18T08:25:00.000Z,4,

In [0]:
print(
    "Silver Candidate rows:",
    spark.table("silver_collisions_candidate").count()
)

Silver Candidate rows: 101527


In [0]:
from pyspark.sql.functions import *

vehicles = spark.table("bronze_vehicles")

print("Bronze vehicles table loaded successfully")

Bronze vehicles table loaded successfully


In [0]:
vehicle_bronze_count = vehicles.count()

print("Total Bronze vehicle records:", vehicle_bronze_count)

Total Bronze vehicle records: 183514


In [0]:
display(vehicles.limit(10))

collision_index,collision_year,collision_ref_no,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre_historic,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane_historic,vehicle_location_restricted_lane,junction_location,skidding_and_overturning,hit_object_in_carriageway,vehicle_leaving_carriageway,hit_object_off_carriageway,first_point_of_impact,vehicle_left_hand_drive,journey_purpose_of_driver_historic,journey_purpose_of_driver,sex_of_driver,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding,ingested_at,source_file
2024471526976,2024,471526976,2,4,0,-1,19,4,8,-1,0,8,0,0,0,0,3,1,-1,9,1,23,5,234,1,16,MAKE AND MODEL REDACTED,8,E01020954,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024471493223,2024,471493223,2,9,0,-1,2,0,0,-1,0,0,0,0,0,0,1,1,-1,6,-1,-1,-1,2198,1,15,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024010504128,2024,010504128,2,9,0,2,2,0,0,0,0,0,0,0,0,0,2,1,6,6,3,-1,-1,2198,1,18,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024302400896,2024,302400896,2,9,0,18,19,1,4,0,0,0,0,0,0,0,1,1,6,6,1,56,9,6496,1,0,MAKE AND MODEL REDACTED,5,E01019782,0,2,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024631472488,2024,631472488,2,90,0,-1,19,7,3,-1,0,8,0,0,0,0,4,1,-1,9,1,55,8,2184,2,0,MAKE AND MODEL REDACTED,7,W01000623,0,4,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024471512096,2024,471512096,1,4,0,-1,19,2,6,-1,0,0,2,0,0,0,4,1,-1,9,1,57,9,449,1,0,MAKE AND MODEL REDACTED,5,E01021074,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024211495435,2024,211495435,1,2,0,-1,3,3,3,-1,0,1,5,0,0,0,2,1,-1,6,2,17,4,49,1,21,MAKE AND MODEL REDACTED,3,E01029752,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024131459645,2024,131459645,1,9,0,-1,10,1,5,-1,0,8,0,0,0,0,3,1,-1,9,1,44,7,1591,1,5,HYUNDAI TUCSON,7,E01011942,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024401457067,2024,401457067,1,9,0,-1,6,7,3,-1,0,0,0,0,0,0,1,1,-1,6,1,26,6,1685,2,9,HYUNDAI TUCSON,10,E01017611,0,3,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024031497068,2024,031497068,1,9,0,-1,19,1,5,-1,0,0,5,0,1,4,1,1,-1,9,1,48,8,1598,1,3,HYUNDAI TUCSON,1,-1,0,5,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv


In [0]:
vehicles.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- collision_year: string (nullable = true)
 |-- collision_ref_no: string (nullable = true)
 |-- vehicle_reference: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- towing_and_articulation: string (nullable = true)
 |-- vehicle_manoeuvre_historic: string (nullable = true)
 |-- vehicle_manoeuvre: string (nullable = true)
 |-- vehicle_direction_from: string (nullable = true)
 |-- vehicle_direction_to: string (nullable = true)
 |-- vehicle_location_restricted_lane_historic: string (nullable = true)
 |-- vehicle_location_restricted_lane: string (nullable = true)
 |-- junction_location: string (nullable = true)
 |-- skidding_and_overturning: string (nullable = true)
 |-- hit_object_in_carriageway: string (nullable = true)
 |-- vehicle_leaving_carriageway: string (nullable = true)
 |-- hit_object_off_carriageway: string (nullable = true)
 |-- first_point_of_impact: string (nullable = true)
 |-- vehicle_left_hand_d

In [0]:
from pyspark.sql.functions import col, trim, expr

vehicles_candidate = (
    vehicles

    # --------------------------------
    # Identifier columns
    # --------------------------------
    .withColumn(
        "collision_index",
        trim(col("collision_index"))
    )
    .withColumn(
        "collision_ref_no",
        trim(col("collision_ref_no"))
    )

    # --------------------------------
    # Integer columns
    # --------------------------------
    .withColumn(
        "collision_year",
        expr("try_cast(collision_year AS INT)")
    )
    .withColumn(
        "vehicle_reference",
        expr("try_cast(vehicle_reference AS INT)")
    )
    .withColumn(
        "vehicle_type",
        expr("try_cast(vehicle_type AS INT)")
    )
    .withColumn(
        "towing_and_articulation",
        expr("try_cast(towing_and_articulation AS INT)")
    )
    .withColumn(
        "vehicle_manoeuvre_historic",
        expr("try_cast(vehicle_manoeuvre_historic AS INT)")
    )
    .withColumn(
        "vehicle_manoeuvre",
        expr("try_cast(vehicle_manoeuvre AS INT)")
    )
    .withColumn(
        "vehicle_direction_from",
        expr("try_cast(vehicle_direction_from AS INT)")
    )
    .withColumn(
        "vehicle_direction_to",
        expr("try_cast(vehicle_direction_to AS INT)")
    )
    .withColumn(
        "vehicle_location_restricted_lane_historic",
        expr("try_cast(vehicle_location_restricted_lane_historic AS INT)")
    )
    .withColumn(
        "vehicle_location_restricted_lane",
        expr("try_cast(vehicle_location_restricted_lane AS INT)")
    )
    .withColumn(
        "junction_location",
        expr("try_cast(junction_location AS INT)")
    )
    .withColumn(
        "skidding_and_overturning",
        expr("try_cast(skidding_and_overturning AS INT)")
    )
    .withColumn(
        "hit_object_in_carriageway",
        expr("try_cast(hit_object_in_carriageway AS INT)")
    )
    .withColumn(
        "vehicle_leaving_carriageway",
        expr("try_cast(vehicle_leaving_carriageway AS INT)")
    )
    .withColumn(
        "hit_object_off_carriageway",
        expr("try_cast(hit_object_off_carriageway AS INT)")
    )
    .withColumn(
        "first_point_of_impact",
        expr("try_cast(first_point_of_impact AS INT)")
    )
    .withColumn(
        "vehicle_left_hand_drive",
        expr("try_cast(vehicle_left_hand_drive AS INT)")
    )
    .withColumn(
        "journey_purpose_of_driver_historic",
        expr("try_cast(journey_purpose_of_driver_historic AS INT)")
    )
    .withColumn(
        "journey_purpose_of_driver",
        expr("try_cast(journey_purpose_of_driver AS INT)")
    )
    .withColumn(
        "sex_of_driver",
        expr("try_cast(sex_of_driver AS INT)")
    )
    .withColumn(
        "age_of_driver",
        expr("try_cast(age_of_driver AS INT)")
    )
    .withColumn(
        "age_band_of_driver",
        expr("try_cast(age_band_of_driver AS INT)")
    )
    .withColumn(
        "engine_capacity_cc",
        expr("try_cast(engine_capacity_cc AS INT)")
    )
    .withColumn(
        "propulsion_code",
        expr("try_cast(propulsion_code AS INT)")
    )
    .withColumn(
        "age_of_vehicle",
        expr("try_cast(age_of_vehicle AS INT)")
    )
    .withColumn(
        "driver_imd_decile",
        expr("try_cast(driver_imd_decile AS INT)")
    )
    .withColumn(
        "escooter_flag",
        expr("try_cast(escooter_flag AS INT)")
    )
    .withColumn(
        "driver_distance_banding",
        expr("try_cast(driver_distance_banding AS INT)")
    )

    # --------------------------------
    # Text standardisation
    # --------------------------------
    .withColumn(
        "generic_make_model",
        trim(col("generic_make_model"))
    )
    .withColumn(
        "lsoa_of_driver",
        trim(col("lsoa_of_driver"))
    )
)

print("Vehicle Silver Candidate created successfully")

Vehicle Silver Candidate created successfully


In [0]:
vehicles_candidate.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- collision_year: integer (nullable = true)
 |-- collision_ref_no: string (nullable = true)
 |-- vehicle_reference: integer (nullable = true)
 |-- vehicle_type: integer (nullable = true)
 |-- towing_and_articulation: integer (nullable = true)
 |-- vehicle_manoeuvre_historic: integer (nullable = true)
 |-- vehicle_manoeuvre: integer (nullable = true)
 |-- vehicle_direction_from: integer (nullable = true)
 |-- vehicle_direction_to: integer (nullable = true)
 |-- vehicle_location_restricted_lane_historic: integer (nullable = true)
 |-- vehicle_location_restricted_lane: integer (nullable = true)
 |-- junction_location: integer (nullable = true)
 |-- skidding_and_overturning: integer (nullable = true)
 |-- hit_object_in_carriageway: integer (nullable = true)
 |-- vehicle_leaving_carriageway: integer (nullable = true)
 |-- hit_object_off_carriageway: integer (nullable = true)
 |-- first_point_of_impact: integer (nullable = true)
 |-- veh

In [0]:
vehicles_candidate = vehicles_candidate.withColumn(
    "vehicle_key",
    concat_ws(
        "_",
        col("collision_index"),
        col("vehicle_reference").cast("string")
    )
)

print("vehicle_key created successfully")

vehicle_key created successfully


In [0]:
display(
    vehicles_candidate.select(
        "collision_index",
        "vehicle_reference",
        "vehicle_key"
    ).limit(20)
)

collision_index,vehicle_reference,vehicle_key
2024471526976,2,2024471526976_2
2024471493223,2,2024471493223_2
2024010504128,2,2024010504128_2
2024302400896,2,2024302400896_2
2024631472488,2,2024631472488_2
2024471512096,1,2024471512096_1
2024211495435,1,2024211495435_1
2024131459645,1,2024131459645_1
2024401457067,1,2024401457067_1
2024031497068,1,2024031497068_1


In [0]:
vehicle_candidate_count = vehicles_candidate.count()

print("Bronze vehicle count   :", vehicle_bronze_count)
print("Candidate vehicle count:", vehicle_candidate_count)
print(
    "Counts match            :",
    vehicle_bronze_count == vehicle_candidate_count
)


Bronze vehicle count   : 183514
Candidate vehicle count: 183514
Counts match            : True


In [0]:
duplicate_vehicle_keys = (
    vehicles_candidate
    .groupBy("vehicle_key")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_vehicle_keys)

vehicle_key,count


In [0]:
display(
    vehicles_candidate
    .filter(
        col("collision_index").isNull()
        | col("vehicle_reference").isNull()
    )
    .select(
        "collision_index",
        "vehicle_reference",
        "vehicle_key"
    )
)

collision_index,vehicle_reference,vehicle_key


In [0]:
display(vehicles_candidate.limit(20))

collision_index,collision_year,collision_ref_no,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre_historic,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane_historic,vehicle_location_restricted_lane,junction_location,skidding_and_overturning,hit_object_in_carriageway,vehicle_leaving_carriageway,hit_object_off_carriageway,first_point_of_impact,vehicle_left_hand_drive,journey_purpose_of_driver_historic,journey_purpose_of_driver,sex_of_driver,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding,ingested_at,source_file,vehicle_key
2024471526976,2024,471526976,2,4,0,-1,19,4,8,-1,0,8,0,0,0,0,3,1,-1,9,1,23,5,234,1,16,MAKE AND MODEL REDACTED,8,E01020954,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471526976_2
2024471493223,2024,471493223,2,9,0,-1,2,0,0,-1,0,0,0,0,0,0,1,1,-1,6,-1,-1,-1,2198,1,15,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471493223_2
2024010504128,2024,010504128,2,9,0,2,2,0,0,0,0,0,0,0,0,0,2,1,6,6,3,-1,-1,2198,1,18,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024010504128_2
2024302400896,2024,302400896,2,9,0,18,19,1,4,0,0,0,0,0,0,0,1,1,6,6,1,56,9,6496,1,0,MAKE AND MODEL REDACTED,5,E01019782,0,2,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024302400896_2
2024631472488,2024,631472488,2,90,0,-1,19,7,3,-1,0,8,0,0,0,0,4,1,-1,9,1,55,8,2184,2,0,MAKE AND MODEL REDACTED,7,W01000623,0,4,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024631472488_2
2024471512096,2024,471512096,1,4,0,-1,19,2,6,-1,0,0,2,0,0,0,4,1,-1,9,1,57,9,449,1,0,MAKE AND MODEL REDACTED,5,E01021074,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471512096_1
2024211495435,2024,211495435,1,2,0,-1,3,3,3,-1,0,1,5,0,0,0,2,1,-1,6,2,17,4,49,1,21,MAKE AND MODEL REDACTED,3,E01029752,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024211495435_1
2024131459645,2024,131459645,1,9,0,-1,10,1,5,-1,0,8,0,0,0,0,3,1,-1,9,1,44,7,1591,1,5,HYUNDAI TUCSON,7,E01011942,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024131459645_1
2024401457067,2024,401457067,1,9,0,-1,6,7,3,-1,0,0,0,0,0,0,1,1,-1,6,1,26,6,1685,2,9,HYUNDAI TUCSON,10,E01017611,0,3,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024401457067_1
2024031497068,2024,031497068,1,9,0,-1,19,1,5,-1,0,0,5,0,1,4,1,1,-1,9,1,48,8,1598,1,3,HYUNDAI TUCSON,1,-1,0,5,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024031497068_1


In [0]:
vehicles_candidate.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_vehicles_candidate")

print("silver_vehicles_candidate created successfully")

silver_vehicles_candidate created successfully


In [0]:
display(
    spark.table("silver_vehicles_candidate").limit(10)
)

collision_index,collision_year,collision_ref_no,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre_historic,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane_historic,vehicle_location_restricted_lane,junction_location,skidding_and_overturning,hit_object_in_carriageway,vehicle_leaving_carriageway,hit_object_off_carriageway,first_point_of_impact,vehicle_left_hand_drive,journey_purpose_of_driver_historic,journey_purpose_of_driver,sex_of_driver,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding,ingested_at,source_file,vehicle_key
2024471526976,2024,471526976,2,4,0,-1,19,4,8,-1,0,8,0,0,0,0,3,1,-1,9,1,23,5,234,1,16,MAKE AND MODEL REDACTED,8,E01020954,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471526976_2
2024471493223,2024,471493223,2,9,0,-1,2,0,0,-1,0,0,0,0,0,0,1,1,-1,6,-1,-1,-1,2198,1,15,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471493223_2
2024010504128,2024,010504128,2,9,0,2,2,0,0,0,0,0,0,0,0,0,2,1,6,6,3,-1,-1,2198,1,18,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024010504128_2
2024302400896,2024,302400896,2,9,0,18,19,1,4,0,0,0,0,0,0,0,1,1,6,6,1,56,9,6496,1,0,MAKE AND MODEL REDACTED,5,E01019782,0,2,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024302400896_2
2024631472488,2024,631472488,2,90,0,-1,19,7,3,-1,0,8,0,0,0,0,4,1,-1,9,1,55,8,2184,2,0,MAKE AND MODEL REDACTED,7,W01000623,0,4,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024631472488_2
2024471512096,2024,471512096,1,4,0,-1,19,2,6,-1,0,0,2,0,0,0,4,1,-1,9,1,57,9,449,1,0,MAKE AND MODEL REDACTED,5,E01021074,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471512096_1
2024211495435,2024,211495435,1,2,0,-1,3,3,3,-1,0,1,5,0,0,0,2,1,-1,6,2,17,4,49,1,21,MAKE AND MODEL REDACTED,3,E01029752,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024211495435_1
2024131459645,2024,131459645,1,9,0,-1,10,1,5,-1,0,8,0,0,0,0,3,1,-1,9,1,44,7,1591,1,5,HYUNDAI TUCSON,7,E01011942,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024131459645_1
2024401457067,2024,401457067,1,9,0,-1,6,7,3,-1,0,0,0,0,0,0,1,1,-1,6,1,26,6,1685,2,9,HYUNDAI TUCSON,10,E01017611,0,3,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024401457067_1
2024031497068,2024,031497068,1,9,0,-1,19,1,5,-1,0,0,5,0,1,4,1,1,-1,9,1,48,8,1598,1,3,HYUNDAI TUCSON,1,-1,0,5,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024031497068_1


In [0]:
print(
    "Silver vehicle rows:",
    spark.table("silver_vehicles_candidate").count()
)

Silver vehicle rows: 183514


In [0]:
final_bronze_count = spark.table("bronze_vehicles").count()
final_silver_count = spark.table("silver_vehicles_candidate").count()

print("Bronze vehicles :", final_bronze_count)
print("Silver vehicles :", final_silver_count)
print("Counts match    :", final_bronze_count == final_silver_count)

Bronze vehicles : 183514
Silver vehicles : 183514
Counts match    : True


In [0]:
%sql
DESCRIBE bronze_casualties

col_name,data_type,comment
collision_index,string,null
collision_year,string,null
collision_ref_no,string,null
vehicle_reference,string,null
casualty_reference,string,null
casualty_class,string,null
sex_of_casualty,string,null
age_of_casualty,string,null
age_band_of_casualty,string,null
casualty_severity,string,null


In [0]:
from pyspark.sql.functions import *

casualties = spark.table("bronze_casualties")

print("Bronze casualties table loaded successfully")

Bronze casualties table loaded successfully


In [0]:
casualty_bronze_count = casualties.count()

print("Total Bronze casualty records:", casualty_bronze_count)

Total Bronze casualty records: 128272


In [0]:
display(casualties.limit(10))

collision_index,collision_year,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding,ingested_at,source_file
2024991534042,2024,991534042,1,1,3,2,39,7,3,1,3,0,0,-1,0,-1,-1,3,1,0,1,-1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024430342145,2024,430342145,2,1,1,2,25,5,3,0,0,0,0,0,1,4,E01028532,-1,0,0.02417,0.97583,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024471442093,2024,471442093,2,1,1,1,49,8,3,0,0,0,0,0,1,8,E01031491,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024101511084,2024,101511084,1,1,1,2,39,7,3,0,0,0,0,0,9,1,E01008859,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024161531062,2024,161531062,1,1,1,1,59,9,3,0,0,0,0,0,1,5,E01012902,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024421434943,2024,421434943,2,1,1,1,62,9,2,0,0,0,0,0,3,5,E01034081,5,1,1,0,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024010533954,2024,010533954,1,1,1,1,44,7,3,0,0,0,0,0,5,5,E01021790,3,1,0,1,3,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024131453448,2024,131453448,1,2,2,2,35,6,2,0,0,1,0,0,9,4,E01011447,7,1,1,0,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024010511909,2024,010511909,2,1,1,1,34,6,3,0,0,0,0,0,3,5,E01032741,3,1,0,1,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024070726166,2024,070726166,2,1,1,1,58,9,2,0,0,0,0,0,3,8,E01014131,7,1,1,0,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv


In [0]:
casualties.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- collision_year: string (nullable = true)
 |-- collision_ref_no: string (nullable = true)
 |-- vehicle_reference: string (nullable = true)
 |-- casualty_reference: string (nullable = true)
 |-- casualty_class: string (nullable = true)
 |-- sex_of_casualty: string (nullable = true)
 |-- age_of_casualty: string (nullable = true)
 |-- age_band_of_casualty: string (nullable = true)
 |-- casualty_severity: string (nullable = true)
 |-- pedestrian_location: string (nullable = true)
 |-- pedestrian_movement: string (nullable = true)
 |-- car_passenger: string (nullable = true)
 |-- bus_or_coach_passenger: string (nullable = true)
 |-- pedestrian_road_maintenance_worker: string (nullable = true)
 |-- casualty_type: string (nullable = true)
 |-- casualty_imd_decile: string (nullable = true)
 |-- lsoa_of_casualty: string (nullable = true)
 |-- enhanced_casualty_severity: string (nullable = true)
 |-- casualty_injury_based: string (nullable 

In [0]:
from pyspark.sql.functions import col, trim, expr

casualties_candidate = (
    casualties

    # --------------------------------
    # Identifier columns
    # --------------------------------
    .withColumn(
        "collision_index",
        trim(col("collision_index"))
    )
    .withColumn(
        "collision_ref_no",
        trim(col("collision_ref_no"))
    )

    # --------------------------------
    # Numeric columns
    # --------------------------------
    .withColumn(
        "collision_year",
        expr("try_cast(collision_year AS INT)")
    )
    .withColumn(
        "vehicle_reference",
        expr("try_cast(vehicle_reference AS INT)")
    )
    .withColumn(
        "casualty_reference",
        expr("try_cast(casualty_reference AS INT)")
    )
    .withColumn(
        "casualty_class",
        expr("try_cast(casualty_class AS INT)")
    )
    .withColumn(
        "sex_of_casualty",
        expr("try_cast(sex_of_casualty AS INT)")
    )
    .withColumn(
        "age_of_casualty",
        expr("try_cast(age_of_casualty AS INT)")
    )
    .withColumn(
        "age_band_of_casualty",
        expr("try_cast(age_band_of_casualty AS INT)")
    )
    .withColumn(
        "casualty_severity",
        expr("try_cast(casualty_severity AS INT)")
    )
    .withColumn(
        "pedestrian_location",
        expr("try_cast(pedestrian_location AS INT)")
    )
    .withColumn(
        "pedestrian_movement",
        expr("try_cast(pedestrian_movement AS INT)")
    )
    .withColumn(
        "car_passenger",
        expr("try_cast(car_passenger AS INT)")
    )
    .withColumn(
        "bus_or_coach_passenger",
        expr("try_cast(bus_or_coach_passenger AS INT)")
    )
    .withColumn(
        "pedestrian_road_maintenance_worker",
        expr("try_cast(pedestrian_road_maintenance_worker AS INT)")
    )
    .withColumn(
        "casualty_type",
        expr("try_cast(casualty_type AS INT)")
    )
    .withColumn(
        "casualty_imd_decile",
        expr("try_cast(casualty_imd_decile AS INT)")
    )
    .withColumn(
        "enhanced_casualty_severity",
        expr("try_cast(enhanced_casualty_severity AS INT)")
    )
    .withColumn(
        "casualty_injury_based",
        expr("try_cast(casualty_injury_based AS INT)")
    )
    .withColumn(
        "casualty_adjusted_severity_serious",
        expr("try_cast(casualty_adjusted_severity_serious AS INT)")
    )
    .withColumn(
        "casualty_adjusted_severity_slight",
        expr("try_cast(casualty_adjusted_severity_slight AS INT)")
    )
    .withColumn(
        "casualty_distance_banding",
        expr("try_cast(casualty_distance_banding AS INT)")
    )

    # --------------------------------
    # Text standardisation
    # --------------------------------
    .withColumn(
        "lsoa_of_casualty",
        trim(col("lsoa_of_casualty"))
    )
)

print("Casualty Silver Candidate created successfully")

Casualty Silver Candidate created successfully


In [0]:
casualties_candidate.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- collision_year: integer (nullable = true)
 |-- collision_ref_no: string (nullable = true)
 |-- vehicle_reference: integer (nullable = true)
 |-- casualty_reference: integer (nullable = true)
 |-- casualty_class: integer (nullable = true)
 |-- sex_of_casualty: integer (nullable = true)
 |-- age_of_casualty: integer (nullable = true)
 |-- age_band_of_casualty: integer (nullable = true)
 |-- casualty_severity: integer (nullable = true)
 |-- pedestrian_location: integer (nullable = true)
 |-- pedestrian_movement: integer (nullable = true)
 |-- car_passenger: integer (nullable = true)
 |-- bus_or_coach_passenger: integer (nullable = true)
 |-- pedestrian_road_maintenance_worker: integer (nullable = true)
 |-- casualty_type: integer (nullable = true)
 |-- casualty_imd_decile: integer (nullable = true)
 |-- lsoa_of_casualty: string (nullable = true)
 |-- enhanced_casualty_severity: integer (nullable = true)
 |-- casualty_injury_based: i

In [0]:
casualties_candidate = casualties_candidate.withColumn(
    "casualty_key",
    concat_ws(
        "_",
        col("collision_index"),
        col("casualty_reference").cast("string")
    )
)

print("casualty_key created successfully")

casualty_key created successfully


In [0]:
display(
    casualties_candidate.select(
        "collision_index",
        "casualty_reference",
        "casualty_key"
    ).limit(20)
)

collision_index,casualty_reference,casualty_key
2024991534042,1,2024991534042_1
2024430342145,1,2024430342145_1
2024471442093,1,2024471442093_1
2024101511084,1,2024101511084_1
2024161531062,1,2024161531062_1
2024421434943,1,2024421434943_1
2024010533954,1,2024010533954_1
2024131453448,2,2024131453448_2
2024010511909,1,2024010511909_1
2024070726166,1,2024070726166_1


In [0]:
casualties_candidate = casualties_candidate.withColumn(
    "casualty_age_group",
    when(col("age_of_casualty").isNull(), "Unknown")
    .when(col("age_of_casualty") < 18, "Under 18")
    .when(col("age_of_casualty").between(18, 30), "18-30")
    .when(col("age_of_casualty").between(31, 50), "31-50")
    .when(col("age_of_casualty").between(51, 70), "51-70")
    .when(col("age_of_casualty") > 70, "71+")
    .otherwise("Unknown")
)

In [0]:
casualties_candidate = casualties_candidate.withColumn(
    "vulnerable_road_user_flag",
    when(col("casualty_class") == 3, 1)
    .otherwise(0)
)

In [0]:
casualties_candidate = casualties_candidate.withColumn(
    "severe_casualty_flag",
    when(col("casualty_severity").isin(1, 2), 1)
    .otherwise(0)
)

In [0]:
display(
    casualties_candidate.select(
        "collision_index",
        "vehicle_reference",
        "casualty_reference",
        "casualty_key",
        "age_of_casualty",
        "casualty_age_group",
        "casualty_class",
        "vulnerable_road_user_flag",
        "casualty_severity",
        "severe_casualty_flag"
    ).limit(20)
)

collision_index,vehicle_reference,casualty_reference,casualty_key,age_of_casualty,casualty_age_group,casualty_class,vulnerable_road_user_flag,casualty_severity,severe_casualty_flag
2024991534042,1,1,2024991534042_1,39,31-50,3,1,3,0
2024430342145,2,1,2024430342145_1,25,18-30,1,0,3,0
2024471442093,2,1,2024471442093_1,49,31-50,1,0,3,0
2024101511084,1,1,2024101511084_1,39,31-50,1,0,3,0
2024161531062,1,1,2024161531062_1,59,51-70,1,0,3,0
2024421434943,2,1,2024421434943_1,62,51-70,1,0,2,1
2024010533954,1,1,2024010533954_1,44,31-50,1,0,3,0
2024131453448,1,2,2024131453448_2,35,31-50,2,0,2,1
2024010511909,2,1,2024010511909_1,34,31-50,1,0,3,0
2024070726166,2,1,2024070726166_1,58,51-70,1,0,2,1


In [0]:
casualty_candidate_count = casualties_candidate.count()

print("Bronze casualty count   :", casualty_bronze_count)
print("Candidate casualty count:", casualty_candidate_count)
print(
    "Counts match             :",
    casualty_bronze_count == casualty_candidate_count
)

Bronze casualty count   : 128272
Candidate casualty count: 128272
Counts match             : True


In [0]:
duplicate_casualty_keys = (
    casualties_candidate
    .groupBy("casualty_key")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_casualty_keys)

casualty_key,count


In [0]:
display(
    casualties_candidate
    .filter(
        col("collision_index").isNull()
        | col("casualty_reference").isNull()
    )
    .select(
        "collision_index",
        "casualty_reference",
        "casualty_key"
    )
)

collision_index,casualty_reference,casualty_key


In [0]:
display(casualties_candidate.limit(20))

collision_index,collision_year,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding,ingested_at,source_file,casualty_key,casualty_age_group,vulnerable_road_user_flag,severe_casualty_flag
2024991534042,2024,991534042,1,1,3,2,39,7,3,1,3,0,0,-1,0,-1,-1,3,1,0,1,-1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024991534042_1,31-50,1,0
2024430342145,2024,430342145,2,1,1,2,25,5,3,0,0,0,0,0,1,4,E01028532,-1,0,null,null,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024430342145_1,18-30,0,0
2024471442093,2024,471442093,2,1,1,1,49,8,3,0,0,0,0,0,1,8,E01031491,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024471442093_1,31-50,0,0
2024101511084,2024,101511084,1,1,1,2,39,7,3,0,0,0,0,0,9,1,E01008859,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024101511084_1,31-50,0,0
2024161531062,2024,161531062,1,1,1,1,59,9,3,0,0,0,0,0,1,5,E01012902,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024161531062_1,51-70,0,0
2024421434943,2024,421434943,2,1,1,1,62,9,2,0,0,0,0,0,3,5,E01034081,5,1,1,0,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024421434943_1,51-70,0,1
2024010533954,2024,010533954,1,1,1,1,44,7,3,0,0,0,0,0,5,5,E01021790,3,1,0,1,3,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024010533954_1,31-50,0,0
2024131453448,2024,131453448,1,2,2,2,35,6,2,0,0,1,0,0,9,4,E01011447,7,1,1,0,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024131453448_2,31-50,0,1
2024010511909,2024,010511909,2,1,1,1,34,6,3,0,0,0,0,0,3,5,E01032741,3,1,0,1,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024010511909_1,31-50,0,0
2024070726166,2024,070726166,2,1,1,1,58,9,2,0,0,0,0,0,3,8,E01014131,7,1,1,0,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024070726166_1,51-70,0,1


In [0]:
casualties_candidate.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_casualties_candidate")

print("silver_casualties_candidate created successfully")

silver_casualties_candidate created successfully


In [0]:
display(
    spark.table("silver_casualties_candidate").limit(10)
)

collision_index,collision_year,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding,ingested_at,source_file,casualty_key,casualty_age_group,vulnerable_road_user_flag,severe_casualty_flag
2024991534042,2024,991534042,1,1,3,2,39,7,3,1,3,0,0,-1,0,-1,-1,3,1,0,1,-1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024991534042_1,31-50,1,0
2024430342145,2024,430342145,2,1,1,2,25,5,3,0,0,0,0,0,1,4,E01028532,-1,0,null,null,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024430342145_1,18-30,0,0
2024471442093,2024,471442093,2,1,1,1,49,8,3,0,0,0,0,0,1,8,E01031491,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024471442093_1,31-50,0,0
2024101511084,2024,101511084,1,1,1,2,39,7,3,0,0,0,0,0,9,1,E01008859,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024101511084_1,31-50,0,0
2024161531062,2024,161531062,1,1,1,1,59,9,3,0,0,0,0,0,1,5,E01012902,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024161531062_1,51-70,0,0
2024421434943,2024,421434943,2,1,1,1,62,9,2,0,0,0,0,0,3,5,E01034081,5,1,1,0,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024421434943_1,51-70,0,1
2024010533954,2024,010533954,1,1,1,1,44,7,3,0,0,0,0,0,5,5,E01021790,3,1,0,1,3,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024010533954_1,31-50,0,0
2024131453448,2024,131453448,1,2,2,2,35,6,2,0,0,1,0,0,9,4,E01011447,7,1,1,0,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024131453448_2,31-50,0,1
2024010511909,2024,010511909,2,1,1,1,34,6,3,0,0,0,0,0,3,5,E01032741,3,1,0,1,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024010511909_1,31-50,0,0
2024070726166,2024,070726166,2,1,1,1,58,9,2,0,0,0,0,0,3,8,E01014131,7,1,1,0,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024070726166_1,51-70,0,1


In [0]:
print(
    "Silver casualty rows:",
    spark.table("silver_casualties_candidate").count()
)

Silver casualty rows: 128272


In [0]:
final_bronze_count = spark.table("bronze_casualties").count()
final_silver_count = spark.table("silver_casualties_candidate").count()

print("Bronze casualties :", final_bronze_count)
print("Silver casualties :", final_silver_count)
print(
    "Counts match      :",
    final_bronze_count == final_silver_count
)

Bronze casualties : 128272
Silver casualties : 128272
Counts match      : True


In [0]:
%sql
show tables;

database,tableName,isTemporary
default,bronze_casualties,false
default,bronze_collision_lookup,false
default,bronze_collisions,false
default,bronze_vehicles,false
default,roadsafe_week03_bronze_demo_collisions,false
default,roadsafe_week03_lineage_demo_view,false
default,silver_casualties_candidate,false
default,silver_collisions_candidate,false
default,silver_vehicles_candidate,false


In [0]:
%sql
DESCRIBE silver_collisions_candidate;

col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
location_easting_osgr,int,null
location_northing_osgr,int,null
longitude,double,null
latitude,double,null
police_force,int,null
collision_severity,int,null
number_of_vehicles,int,null


In [0]:
%sql
SHOW TABLES LIKE 'silver_*_candidate';


In [0]:
%sql
DESCRIBE silver_vehicles_candidate;


col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
vehicle_reference,int,null
vehicle_type,int,null
towing_and_articulation,int,null
vehicle_manoeuvre_historic,int,null
vehicle_manoeuvre,int,null
vehicle_direction_from,int,null
vehicle_direction_to,int,null


In [0]:
%sql
DESCRIBE silver_casualties_candidate;

col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
vehicle_reference,int,null
casualty_reference,int,null
casualty_class,int,null
sex_of_casualty,int,null
age_of_casualty,int,null
age_band_of_casualty,int,null
casualty_severity,int,null


In [0]:
%sql
SELECT
    current_catalog() AS catalog,
    current_schema() AS schema;

catalog,schema
workspace,default


In [0]:
%sql
SHOW TABLES LIKE 'silver_*_candidate';

database,tableName,isTemporary
default,silver_collisions_candidate,false


In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,silver_collisions_candidate,false


In [0]:
%sql
show tables;
describe silver_casualties_candidate;

col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
vehicle_reference,int,null
casualty_reference,int,null
casualty_class,int,null
sex_of_casualty,int,null
age_of_casualty,int,null
age_band_of_casualty,int,null
casualty_severity,int,null


In [0]:
%sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = current_schema()
  AND LOWER(table_name) LIKE '%candidate%'
ORDER BY table_name;

table_name
silver_casualties_candidate
silver_collisions_candidate
silver_vehicles_candidate


In [0]:
%sql
SELECT
    collision_index,
    source_file
FROM silver_collisions_candidate
LIMIT 10;

collision_index,source_file
202417M119024,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
202417S312124,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024070110901,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024041446676,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024041478641,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024161425653,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024161438773,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024122400772,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024302400537,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv
2024302400805,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv


In [0]:
%sql
SELECT
    collision_index,
    vehicle_reference,
    vehicle_key,
    source_file
FROM silver_vehicles_candidate
LIMIT 10;

collision_index,vehicle_reference,vehicle_key,source_file
2024471526976,2,2024471526976_2,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024471493223,2,2024471493223_2,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024010504128,2,2024010504128_2,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024302400896,2,2024302400896_2,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024631472488,2,2024631472488_2,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024471512096,1,2024471512096_1,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024211495435,1,2024211495435_1,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024131459645,1,2024131459645_1,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024401457067,1,2024401457067_1,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv
2024031497068,1,2024031497068_1,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv


In [0]:
%sql
SELECT
    collision_index,
    casualty_reference,
    casualty_key,
    source_file
FROM silver_casualties_candidate
LIMIT 10;

collision_index,casualty_reference,casualty_key,source_file
2024991534042,1,2024991534042_1,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024430342145,1,2024430342145_1,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024471442093,1,2024471442093_1,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024101511084,1,2024101511084_1,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024161531062,1,2024161531062_1,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024421434943,1,2024421434943_1,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024010533954,1,2024010533954_1,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024131453448,2,2024131453448_2,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024010511909,1,2024010511909_1,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv
2024070726166,1,2024070726166_1,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_collision_ids AS
SELECT
    collision_index,
    COUNT(*) AS occurrences
FROM silver_collisions_candidate
WHERE collision_index IS NOT NULL
  AND TRIM(collision_index) <> ''
GROUP BY collision_index
HAVING COUNT(*) > 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW collisions_dq01 AS
SELECT
    c.*,
    CASE
        WHEN c.collision_index IS NULL
          OR TRIM(c.collision_index) = ''
            THEN 'FAIL'

        WHEN d.collision_index IS NOT NULL
            THEN 'FAIL'

        ELSE 'PASS'
    END AS dq01_status
FROM silver_collisions_candidate c
LEFT JOIN duplicate_collision_ids d
    ON c.collision_index = d.collision_index;

In [0]:
%sql
SELECT
    dq01_status,
    COUNT(*) AS record_count
FROM collisions_dq01
GROUP BY dq01_status;

dq01_status,record_count
PASS,100427
FAIL,1100


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_vehicle_keys AS
SELECT
    collision_index,
    vehicle_reference,
    COUNT(*) AS occurrences
FROM silver_vehicles_candidate
WHERE collision_index IS NOT NULL
  AND vehicle_reference IS NOT NULL
GROUP BY
    collision_index,
    vehicle_reference
HAVING COUNT(*) > 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vehicles_dq02 AS
SELECT
    v.*,
    CASE
        WHEN v.collision_index IS NULL
          OR TRIM(v.collision_index) = ''
          OR v.vehicle_reference IS NULL
            THEN 'FAIL'

        WHEN d.collision_index IS NOT NULL
            THEN 'FAIL'

        WHEN c.collision_index IS NULL
            THEN 'FAIL'

        ELSE 'PASS'
    END AS dq02_status

FROM silver_vehicles_candidate v

LEFT JOIN duplicate_vehicle_keys d
    ON v.collision_index = d.collision_index
   AND v.vehicle_reference = d.vehicle_reference

LEFT JOIN (
    SELECT DISTINCT collision_index
    FROM silver_collisions_candidate
) c
    ON v.collision_index = c.collision_index;

In [0]:
%sql
SELECT
    dq02_status,
    COUNT(*) AS record_count
FROM vehicles_dq02
GROUP BY dq02_status;

dq02_status,record_count
FAIL,1500
PASS,182014


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_casualty_keys AS
SELECT
    collision_index,
    casualty_reference,
    COUNT(*) AS occurrences
FROM silver_casualties_candidate
WHERE collision_index IS NOT NULL
  AND casualty_reference IS NOT NULL
GROUP BY
    collision_index,
    casualty_reference
HAVING COUNT(*) > 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW casualties_dq03 AS
SELECT
    ca.*,
    CASE
        WHEN ca.collision_index IS NULL
          OR TRIM(ca.collision_index) = ''
          OR ca.casualty_reference IS NULL
            THEN 'FAIL'

        WHEN d.collision_index IS NOT NULL
            THEN 'FAIL'

        WHEN c.collision_index IS NULL
            THEN 'FAIL'

        WHEN ca.vehicle_reference IS NOT NULL
         AND v.vehicle_reference IS NULL
            THEN 'FAIL'

        ELSE 'PASS'
    END AS dq03_status

FROM silver_casualties_candidate ca

LEFT JOIN duplicate_casualty_keys d
    ON ca.collision_index = d.collision_index
   AND ca.casualty_reference = d.casualty_reference

LEFT JOIN (
    SELECT DISTINCT collision_index
    FROM silver_collisions_candidate
) c
    ON ca.collision_index = c.collision_index

LEFT JOIN (
    SELECT DISTINCT
        collision_index,
        vehicle_reference
    FROM silver_vehicles_candidate
) v
    ON ca.collision_index = v.collision_index
   AND ca.vehicle_reference = v.vehicle_reference;

In [0]:
%sql
SELECT
    dq03_status,
    COUNT(*) AS record_count
FROM casualties_dq03
GROUP BY dq03_status;

dq03_status,record_count
FAIL,3093
PASS,125179


In [0]:
%sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = current_schema()
  AND (
       LOWER(table_name) LIKE '%lookup%'
       OR LOWER(table_name) LIKE '%reference%'
       OR LOWER(table_name) LIKE '%domain%'
      )
ORDER BY table_name;

table_name
bronze_collision_lookup


In [0]:
%sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = current_schema()
  AND (
       LOWER(table_name) LIKE '%lookup%'
       OR LOWER(table_name) LIKE '%reference%'
       OR LOWER(table_name) LIKE '%domain%'
      )
ORDER BY table_name;

table_name
bronze_collision_lookup


In [0]:
%sql
SELECT
    date,
    time,
    collision_timestamp,
    collision_year
FROM silver_collisions_candidate
LIMIT 20;

date,time,collision_timestamp,collision_year
05/12/2024,16:10,2024-12-05T16:10:00.000Z,2024
22/10/2024,14:56,2024-10-22T14:56:00.000Z,2024
24/01/2024,17:50,2024-01-24T17:50:00.000Z,2024
22/05/2024,17:45,2024-05-22T17:45:00.000Z,2024
11/08/2024,15:36,2024-08-11T15:36:00.000Z,2024
19/03/2024,14:40,2024-03-19T14:40:00.000Z,2024
03/05/2024,08:19,2024-05-03T08:19:00.000Z,2024
30/05/2024,08:29,2024-05-30T08:29:00.000Z,2024
18/04/2024,08:25,2024-04-18T08:25:00.000Z,2024
24/05/2024,17:10,2024-05-24T17:10:00.000Z,2024


In [0]:
%sql
SELECT
    latitude,
    longitude
FROM silver_collisions_candidate
LIMIT 20;

latitude,longitude
54.57219,-1.22722
54.54466,-1.29265
53.41443,-2.53157
53.67359,-2.5163
53.76592,-2.50416
53.52234,-0.09323
53.57841,-0.66503
53.98933,-1.10184
52.89721,-1.42873
52.89648,-1.47138


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW valid_vehicle_counts AS
SELECT
    collision_index,
    COUNT(*) AS valid_vehicle_count
FROM vehicles_dq02
WHERE dq02_status = 'PASS'
GROUP BY collision_index;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW valid_casualty_counts AS
SELECT
    collision_index,
    COUNT(*) AS valid_casualty_count
FROM casualties_dq03
WHERE dq03_status = 'PASS'
GROUP BY collision_index;

In [0]:
%sql
SELECT
    c.collision_index,

    c.number_of_vehicles AS reported_vehicles,

    COALESCE(v.valid_vehicle_count, 0)
        AS valid_vehicles,

    c.number_of_casualties AS reported_casualties,

    COALESCE(ca.valid_casualty_count, 0)
        AS valid_casualties

FROM silver_collisions_candidate c

LEFT JOIN valid_vehicle_counts v
    ON c.collision_index = v.collision_index

LEFT JOIN valid_casualty_counts ca
    ON c.collision_index = ca.collision_index;

collision_index,reported_vehicles,valid_vehicles,reported_casualties,valid_casualties
202417M119024,2,2,1,1
202417S312124,1,0,1,0
2024070110901,2,2,1,1
2024041446676,1,1,1,1
2024041478641,2,2,1,1
2024161425653,2,2,2,2
2024161438773,1,1,1,1
2024122400772,4,4,1,1
2024302400537,2,2,2,2
2024302400805,2,2,1,1


In [0]:
%sql
SELECT *
FROM bronze_collision_lookup
LIMIT 20;

lookup_domain,code,description,note,ingested_at,source_file
police_force,1,Metropolitan Police,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
police_force,3,Cumbria,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
police_force,4,Lancashire,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
police_force,5,Merseyside,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
police_force,6,Greater Manchester,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
police_force,7,Cheshire,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
police_force,10,Northumbria,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
police_force,11,Durham,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
police_force,12,North Yorkshire,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
police_force,13,West Yorkshire,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv


In [0]:
%sql
DESCRIBE bronze_collision_lookup;

col_name,data_type,comment
lookup_domain,string,null
code,string,null
description,string,null
note,string,null
ingested_at,timestamp,null
source_file,string,null


In [0]:
%sql
SELECT DISTINCT lookup_domain
FROM bronze_collision_lookup
ORDER BY lookup_domain;

lookup_domain
carriageway_hazards
carriageway_hazards_historic
collision_injury_based
collision_severity
day_of_week
did_police_officer_attend_scene_of_accident
did_police_officer_attend_scene_of_collision
enhanced_collision_severity
first_road_class
first_road_number


In [0]:
%sql
SELECT
    collision_severity,
    COUNT(*) AS record_count
FROM silver_collisions_candidate
GROUP BY collision_severity
ORDER BY collision_severity;

collision_severity,record_count
1,1505
2,23615
3,75904
8,503


In [0]:
%sql
SELECT
    table_catalog,
    table_schema,
    table_name
FROM information_schema.tables
WHERE LOWER(table_name) LIKE '%candidate%'
ORDER BY table_schema, table_name;

table_catalog,table_schema,table_name
workspace,default,silver_casualties_candidate
workspace,default,silver_collisions_candidate
workspace,default,silver_vehicles_candidate


In [0]:
%sql
SELECT
    casualty_severity,
    COUNT(*) AS record_count
FROM workspace.default.silver_casualties_candidate
GROUP BY casualty_severity
ORDER BY casualty_severity;

casualty_severity,record_count
1,1598
2,25947
3,100227
8,500


In [0]:
%sql
SELECT
    collision_severity,
    COUNT(*) AS record_count
FROM workspace.default.silver_collisions_candidate
GROUP BY collision_severity
ORDER BY collision_severity;

collision_severity,record_count
1,1505
2,23615
3,75904
8,503


In [0]:
%sql
SELECT DISTINCT lookup_domain
FROM workspace.default.bronze_collision_lookup
ORDER BY lookup_domain;

lookup_domain
carriageway_hazards
carriageway_hazards_historic
collision_injury_based
collision_severity
day_of_week
did_police_officer_attend_scene_of_accident
did_police_officer_attend_scene_of_collision
enhanced_collision_severity
first_road_class
first_road_number


In [0]:
%sql
SELECT *
FROM workspace.default.bronze_collision_lookup
WHERE lookup_domain = 'collision_severity'
ORDER BY code;

lookup_domain,code,description,note,ingested_at,source_file
collision_severity,1,Fatal,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
collision_severity,2,Serious,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv
collision_severity,3,Slight,null,2026-07-31T10:01:11.451Z,dbfs:/Volumes/road_safety/default/pg-13/collision_code_lookup_sample.csv


In [0]:
%sql
DESCRIBE workspace.default.bronze_collision_lookup;

col_name,data_type,comment
lookup_domain,string,null
code,string,null
description,string,null
note,string,null
ingested_at,timestamp,null
source_file,string,null


In [0]:
%sql
SELECT DISTINCT lookup_domain
FROM workspace.default.bronze_collision_lookup
WHERE LOWER(lookup_domain) LIKE '%casual%'
   OR LOWER(lookup_domain) LIKE '%severity%'
ORDER BY lookup_domain;

lookup_domain
collision_severity
enhanced_collision_severity


In [0]:
%sql
SELECT
    c.collision_index,
    c.collision_severity,
    l.code AS lookup_code,
    l.description,
    CASE
        WHEN c.collision_severity IS NULL THEN 1
        WHEN l.code IS NULL THEN 1
        ELSE 0
    END AS dq04_collision_severity_fail
FROM workspace.default.silver_collisions_candidate c
LEFT JOIN workspace.default.bronze_collision_lookup l
    ON l.lookup_domain = 'collision_severity'
   AND CAST(l.code AS INT) = c.collision_severity
WHERE c.collision_severity IS NULL
   OR l.code IS NULL;

collision_index,collision_severity,lookup_code,description,dq04_collision_severity_fail
2024221485595,8,null,null,1
2024401539364,8,null,null,1
2024010522574,8,null,null,1
2024201484886,8,null,null,1
2024991530211,8,null,null,1
2024401402005,8,null,null,1
2024010523120,8,null,null,1
2024061446399,8,null,null,1
2024991448097,8,null,null,1
2024430306094,8,null,null,1


In [0]:
%sql
SELECT
    COUNT(*) AS dq04_collision_failures
FROM workspace.default.silver_collisions_candidate c
LEFT JOIN workspace.default.bronze_collision_lookup l
    ON l.lookup_domain = 'collision_severity'
   AND CAST(l.code AS INT) = c.collision_severity
WHERE c.collision_severity IS NULL
   OR l.code IS NULL;

dq04_collision_failures
503


In [0]:
%sql
SELECT
    collision_severity,
    COUNT(*) AS failure_count
FROM workspace.default.silver_collisions_candidate c
LEFT JOIN workspace.default.bronze_collision_lookup l
    ON l.lookup_domain = 'collision_severity'
   AND CAST(l.code AS INT) = c.collision_severity
WHERE c.collision_severity IS NULL
   OR l.code IS NULL
GROUP BY collision_severity
ORDER BY collision_severity;

collision_severity,failure_count
8,503


In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bronze_casualties,false
default,bronze_collision_lookup,false
default,bronze_collisions,false
default,bronze_vehicles,false
default,roadsafe_week03_bronze_demo_collisions,false
default,roadsafe_week03_lineage_demo_view,false
default,silver_casualties_candidate,false
default,silver_collisions_candidate,false
default,silver_vehicles_candidate,false


In [0]:
%sql
SELECT
    casualty_severity,
    COUNT(*) AS record_count
FROM workspace.default.bronze_casualties
GROUP BY casualty_severity
ORDER BY casualty_severity;

casualty_severity,record_count
1,1598
2,25947
3,100227
8,500


In [0]:
%sql
DESCRIBE workspace.default.bronze_casualties;

col_name,data_type,comment
collision_index,string,null
collision_year,string,null
collision_ref_no,string,null
vehicle_reference,string,null
casualty_reference,string,null
casualty_class,string,null
sex_of_casualty,string,null
age_of_casualty,string,null
age_band_of_casualty,string,null
casualty_severity,string,null


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dq04_collision AS
SELECT
    c.*,
    CASE
        WHEN c.collision_severity IS NULL THEN 1
        WHEN l.code IS NULL THEN 1
        ELSE 0
    END AS dq04_collision_fail
FROM workspace.default.silver_collisions_candidate c
LEFT JOIN workspace.default.bronze_collision_lookup l
    ON l.lookup_domain = 'collision_severity'
   AND CAST(l.code AS INT) = c.collision_severity;

In [0]:
%sql
SELECT
    dq04_collision_fail,
    COUNT(*) AS record_count
FROM dq04_collision
GROUP BY dq04_collision_fail
ORDER BY dq04_collision_fail;

dq04_collision_fail,record_count
0,101024
1,503


In [0]:
%sql
SELECT *
FROM dq04_collision
WHERE dq04_collision_fail = 1
LIMIT 20;

collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,local_authority_district,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail_historic,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control_historic,pedestrian_crossing_physical_facilities_historic,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,ingested_at,source_file,collision_timestamp,collision_month,collision_hour,collision_weekday,weekend_flag,time_band,severe_collision_flag,urban_rural_group,dq04_collision_fail
2024221485595,2024,221485595,352117,240176,-2.69979,52.05788,22,8,1,1,30/08/2024,6,20:42,-1,E06000019,E06000019,E06000019,6,0,6,30,-1,0,-1,6,0,-1,-1,13,4,1,1,-1,-1,0,1,1,2,E01013988,1,1,0,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-08-30T20:42:00.000Z,8,20,6,0,18:00-23:59,0,Urban,1
2024401539364,2024,401539364,517096,257950,-0.2878,52.20734,40,8,3,2,01/12/2024,1,15:05,-1,E06000055,E06000055,E06000055,3,428,3,60,-1,0,-1,3,1,-1,-1,0,1,8,2,-1,-1,0,2,1,1,E01017542,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-12-01T15:05:00.000Z,12,15,1,1,12:00-17:59,0,Rural,1
2024010522574,2024,010522574,547683,185584,0.12858,51.54982,1,8,2,1,19/07/2024,6,12:25,-1,E09000002,E09000002,E09000002,3,1153,6,30,6,16,4,6,0,0,0,0,1,1,1,0,0,0,1,1,2,E01000077,-1,0,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-07-19T12:25:00.000Z,7,12,6,0,12:00-17:59,0,Urban,1
2024201484886,2024,201484886,417513,286172,-1.7436,52.4732,20,8,1,1,24/08/2024,7,22:10,-1,E08000029,E08000029,E08000029,6,0,6,30,-1,0,-1,-1,-1,-1,-1,0,4,1,1,-1,-1,0,2,1,2,E01010112,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-08-24T22:10:00.000Z,8,22,7,1,18:00-23:59,0,Rural,1
2024991530211,2024,991530211,313956,677043,-3.38045,55.97863,99,8,1,1,14/12/2024,7,18:48,-1,S12000036,S12000036,S12000036,3,90,3,50,-1,0,-1,-1,-1,-1,-1,0,4,1,2,-1,-1,0,2,1,-1,-1,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-12-14T18:48:00.000Z,12,18,7,1,18:00-23:59,0,Rural,1
2024401402005,2024,401402005,508603,221559,-0.42358,51.88203,40,8,2,1,18/01/2024,5,20:57,-1,E06000032,E06000032,E06000032,3,6,3,30,-1,19,2,3,505,-1,-1,0,4,8,2,-1,-1,0,1,1,2,E01015729,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-01-18T20:57:00.000Z,1,20,5,0,18:00-23:59,0,Urban,1
2024010523120,2024,010523120,522692,168365,-0.23762,51.40106,1,8,1,1,22/07/2024,2,13:05,-1,E09000024,E09000024,E09000024,4,282,9,30,3,13,4,6,0,9,9,99,1,9,9,9,9,99,1,3,2,E01003470,-1,0,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-07-22T13:05:00.000Z,7,13,2,0,12:00-17:59,0,Urban,1
2024061446399,2024,061446399,381228,404904,-2.28473,53.54044,6,8,2,1,22/05/2024,4,16:27,-1,E08000002,E08000002,E08000002,3,665,6,30,-1,13,4,6,0,-1,-1,0,1,2,2,-1,-1,21,1,1,2,E01004971,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-22T16:27:00.000Z,5,16,4,0,12:00-17:59,0,Urban,1
2024991448097,2024,991448097,275073,845117,-4.08511,57.47903,99,8,2,2,25/05/2024,7,08:00,-1,S12000017,S12000017,S12000017,4,851,6,60,-1,16,4,6,0,-1,-1,0,1,1,1,-1,-1,0,2,1,-1,-1,7,1,1,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-25T08:

In [0]:
%sql

SELECT DISTINCT lookup_domain
FROM workspace.default.bronze_collision_lookup
ORDER BY lookup_domain;

lookup_domain
carriageway_hazards
carriageway_hazards_historic
collision_injury_based
collision_severity
day_of_week
did_police_officer_attend_scene_of_accident
did_police_officer_attend_scene_of_collision
enhanced_collision_severity
first_road_class
first_road_number


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq08_collision AS
SELECT
    c.*,

    CASE
        WHEN c.speed_limit IS NOT NULL
             AND sl.code IS NULL THEN 1

        WHEN c.light_conditions IS NOT NULL
             AND lc.code IS NULL THEN 1

        WHEN c.weather_conditions IS NOT NULL
             AND wc.code IS NULL THEN 1

        WHEN c.road_surface_conditions IS NOT NULL
             AND rs.code IS NULL THEN 1

        WHEN c.special_conditions_at_site IS NOT NULL
             AND sc.code IS NULL THEN 1

        WHEN c.carriageway_hazards IS NOT NULL
             AND ch.code IS NULL THEN 1

        ELSE 0
    END AS dq08_collision_fail

FROM workspace.default.silver_collisions_candidate c

LEFT JOIN workspace.default.bronze_collision_lookup sl
    ON sl.lookup_domain = 'speed_limit'
   AND CAST(sl.code AS INT) = c.speed_limit

LEFT JOIN workspace.default.bronze_collision_lookup lc
    ON lc.lookup_domain = 'light_conditions'
   AND CAST(lc.code AS INT) = c.light_conditions

LEFT JOIN workspace.default.bronze_collision_lookup wc
    ON wc.lookup_domain = 'weather_conditions'
   AND CAST(wc.code AS INT) = c.weather_conditions

LEFT JOIN workspace.default.bronze_collision_lookup rs
    ON rs.lookup_domain = 'road_surface_conditions'
   AND CAST(rs.code AS INT) = c.road_surface_conditions

LEFT JOIN workspace.default.bronze_collision_lookup sc
    ON sc.lookup_domain = 'special_conditions_at_site'
   AND CAST(sc.code AS INT) = c.special_conditions_at_site

LEFT JOIN workspace.default.bronze_collision_lookup ch
    ON ch.lookup_domain = 'carriageway_hazards'
   AND CAST(ch.code AS INT) = c.carriageway_hazards;

In [0]:
%sql

SELECT
    speed_limit,
    light_conditions,
    weather_conditions,
    road_surface_conditions,
    special_conditions_at_site,
    carriageway_hazards,
    COUNT(*) AS record_count
FROM dq08_collision
WHERE dq08_collision_fail = 1
GROUP BY
    speed_limit,
    light_conditions,
    weather_conditions,
    road_surface_conditions,
    special_conditions_at_site,
    carriageway_hazards
ORDER BY record_count DESC;

speed_limit,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards,record_count
30,1,1,1,-1,0,16903
30,1,1,1,0,0,8790
20,1,1,1,0,0,6601
30,4,1,1,-1,0,4125
60,1,1,1,-1,0,4015
40,1,1,1,-1,0,2878
20,1,1,1,-1,0,2488
30,4,1,1,0,0,2255
20,4,1,1,0,0,2216
30,1,2,2,-1,0,1977


In [0]:
%sql

SELECT
    dq08_collision_fail,
    COUNT(*) AS record_count
FROM dq08_collision
GROUP BY dq08_collision_fail
ORDER BY dq08_collision_fail;

dq08_collision_fail,record_count
0,3
1,101524


In [0]:
%sql

SELECT
    lookup_domain,
    code,
    COUNT(*) AS cnt
FROM workspace.default.bronze_collision_lookup
WHERE lookup_domain IN (
    'speed_limit',
    'light_conditions',
    'weather_conditions',
    'road_surface_conditions',
    'special_conditions_at_site',
    'carriageway_hazards'
)
GROUP BY lookup_domain, code
ORDER BY lookup_domain, code;

lookup_domain,code,cnt
carriageway_hazards,-1,1
carriageway_hazards,0,1
carriageway_hazards,11,1
carriageway_hazards,12,1
carriageway_hazards,13,1
carriageway_hazards,14,1
carriageway_hazards,15,1
carriageway_hazards,16,1
carriageway_hazards,17,1
carriageway_hazards,18,1


In [0]:
%sql

SELECT
    speed_limit,
    light_conditions,
    weather_conditions,
    road_surface_conditions,
    special_conditions_at_site,
    carriageway_hazards,
    COUNT(*) AS record_count
FROM workspace.default.silver_collisions_candidate
GROUP BY
    speed_limit,
    light_conditions,
    weather_conditions,
    road_surface_conditions,
    special_conditions_at_site,
    carriageway_hazards
ORDER BY record_count DESC
LIMIT 30;

speed_limit,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards,record_count
30,1,1,1,-1,0,16903
30,1,1,1,0,0,8790
20,1,1,1,0,0,6601
30,4,1,1,-1,0,4125
60,1,1,1,-1,0,4015
40,1,1,1,-1,0,2878
20,1,1,1,-1,0,2488
30,4,1,1,0,0,2255
20,4,1,1,0,0,2216
30,1,2,2,-1,0,1977


In [0]:
%sql

SELECT
    lookup_domain,
    code,
    COUNT(*) AS record_count
FROM workspace.default.bronze_collision_lookup
WHERE lookup_domain IN (
    'speed_limit',
    'light_conditions',
    'weather_conditions',
    'road_surface_conditions',
    'special_conditions_at_site',
    'carriageway_hazards'
)
GROUP BY lookup_domain, code
ORDER BY lookup_domain, code;


SELECT
    lookup_domain,
    COUNT(*) AS lookup_count
FROM workspace.default.bronze_collision_lookup
WHERE lookup_domain IN (
    'speed_limit',
    'light_conditions',
    'weather_conditions',
    'road_surface_conditions',
    'special_conditions_at_site',
    'carriageway_hazards'
)
GROUP BY lookup_domain
ORDER BY lookup_domain;

lookup_domain,lookup_count
carriageway_hazards,14
light_conditions,6
road_surface_conditions,9
special_conditions_at_site,10
speed_limit,2
weather_conditions,10


In [0]:
%sql

SELECT
    lookup_domain,
    code,
    description
FROM workspace.default.bronze_collision_lookup
WHERE lookup_domain IN (
    'speed_limit',
    'light_conditions',
    'weather_conditions',
    'road_surface_conditions',
    'special_conditions_at_site',
    'carriageway_hazards'
)
ORDER BY lookup_domain, code;

lookup_domain,code,description
carriageway_hazards,-1,Data missing or out of range
carriageway_hazards,0,None
carriageway_hazards,11,Defective traffic signals
carriageway_hazards,12,Permanent road signing or markings defective or obscured or inadequate
carriageway_hazards,13,Roadworks
carriageway_hazards,14,Oil or diesel
carriageway_hazards,15,Mud
carriageway_hazards,16,Dislodged vehicle load in carriageway
carriageway_hazards,17,Other object in carriageway
carriageway_hazards,18,Involvement with previous collision


In [0]:
%sql

SELECT
    COUNT(*) AS total_records,

    SUM(CASE WHEN speed_limit IS NULL THEN 1 ELSE 0 END) AS speed_null,
    SUM(CASE WHEN light_conditions IS NULL THEN 1 ELSE 0 END) AS light_null,
    SUM(CASE WHEN weather_conditions IS NULL THEN 1 ELSE 0 END) AS weather_null,
    SUM(CASE WHEN road_surface_conditions IS NULL THEN 1 ELSE 0 END) AS surface_null,
    SUM(CASE WHEN special_conditions_at_site IS NULL THEN 1 ELSE 0 END) AS special_null,
    SUM(CASE WHEN carriageway_hazards IS NULL THEN 1 ELSE 0 END) AS hazard_null

FROM workspace.default.silver_collisions_candidate;

total_records,speed_null,light_null,weather_null,surface_null,special_null,hazard_null
101527,0,0,0,0,0,0


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq08_collision AS
SELECT
    c.*,

    CASE
        WHEN c.speed_limit IS NOT NULL
             AND sl.code IS NULL THEN 1

        WHEN c.light_conditions IS NOT NULL
             AND lc.code IS NULL THEN 1

        WHEN c.weather_conditions IS NOT NULL
             AND wc.code IS NULL THEN 1

        WHEN c.road_surface_conditions IS NOT NULL
             AND rs.code IS NULL THEN 1

        WHEN c.special_conditions_at_site IS NOT NULL
             AND sc.code IS NULL THEN 1

        WHEN c.carriageway_hazards IS NOT NULL
             AND ch.code IS NULL THEN 1

        ELSE 0
    END AS dq08_collision_fail

FROM workspace.default.silver_collisions_candidate c

LEFT JOIN workspace.default.bronze_collision_lookup sl
    ON sl.lookup_domain = 'speed_limit'
    AND CAST(sl.code AS INT) = c.speed_limit

LEFT JOIN workspace.default.bronze_collision_lookup lc
    ON lc.lookup_domain = 'light_conditions'
    AND CAST(lc.code AS INT) = c.light_conditions

LEFT JOIN workspace.default.bronze_collision_lookup wc
    ON wc.lookup_domain = 'weather_conditions'
    AND CAST(wc.code AS INT) = c.weather_conditions

LEFT JOIN workspace.default.bronze_collision_lookup rs
    ON rs.lookup_domain = 'road_surface_conditions'
    AND CAST(rs.code AS INT) = c.road_surface_conditions

LEFT JOIN workspace.default.bronze_collision_lookup sc
    ON sc.lookup_domain = 'special_conditions_at_site'
    AND CAST(sc.code AS INT) = c.special_conditions_at_site

LEFT JOIN workspace.default.bronze_collision_lookup ch
    ON ch.lookup_domain = 'carriageway_hazards'
    AND CAST(ch.code AS INT) = c.carriageway_hazards;

In [0]:
%sql

SELECT
    dq08_collision_fail,
    COUNT(*) AS record_count
FROM dq08_collision
GROUP BY dq08_collision_fail
ORDER BY dq08_collision_fail;

dq08_collision_fail,record_count
0,3
1,101524


In [0]:
%sql

SELECT
    lookup_domain,
    code,
    description
FROM workspace.default.bronze_collision_lookup
WHERE lookup_domain IN (
    'speed_limit',
    'light_conditions',
    'weather_conditions',
    'road_surface_conditions',
    'special_conditions_at_site',
    'carriageway_hazards'
)
ORDER BY lookup_domain, code;

lookup_domain,code,description
carriageway_hazards,-1,Data missing or out of range
carriageway_hazards,0,None
carriageway_hazards,11,Defective traffic signals
carriageway_hazards,12,Permanent road signing or markings defective or obscured or inadequate
carriageway_hazards,13,Roadworks
carriageway_hazards,14,Oil or diesel
carriageway_hazards,15,Mud
carriageway_hazards,16,Dislodged vehicle load in carriageway
carriageway_hazards,17,Other object in carriageway
carriageway_hazards,18,Involvement with previous collision


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq08_collision AS
SELECT
    c.*,

    CASE
        -- Speed limit: handle declared unknown/missing separately
        WHEN c.speed_limit IS NOT NULL
             AND c.speed_limit NOT IN (-1, 99)
             AND (
                 c.speed_limit < 20
                 OR c.speed_limit > 70
             )
        THEN 1

        -- Light conditions
        WHEN c.light_conditions IS NOT NULL
             AND c.light_conditions != -1
             AND c.light_conditions NOT IN (1, 4, 5, 6, 7)
        THEN 1

        -- Weather conditions
        WHEN c.weather_conditions IS NOT NULL
             AND c.weather_conditions != -1
             AND c.weather_conditions NOT IN (1, 2, 3, 4, 5, 6, 7, 8, 9)
        THEN 1

        -- Road surface
        WHEN c.road_surface_conditions IS NOT NULL
             AND c.road_surface_conditions != -1
             AND c.road_surface_conditions NOT IN (1, 2, 3, 4, 5, 6, 7, 9)
        THEN 1

        -- Special conditions
        WHEN c.special_conditions_at_site IS NOT NULL
             AND c.special_conditions_at_site != -1
             AND c.special_conditions_at_site NOT IN (0, 1, 2, 3, 4, 5, 6, 7, 9)
        THEN 1

        -- Carriageway hazards
        WHEN c.carriageway_hazards IS NOT NULL
             AND c.carriageway_hazards != -1
             AND c.carriageway_hazards NOT IN (
                 0, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 99
             )
        THEN 1

        ELSE 0
    END AS dq08_collision_fail

FROM workspace.default.silver_collisions_candidate c;

In [0]:
%sql

SELECT
    speed_limit,
    COUNT(*) AS record_count
FROM workspace.default.silver_collisions_candidate
GROUP BY speed_limit
ORDER BY speed_limit;

speed_limit,record_count
-1,3
20,18938
30,50421
40,8975
50,4699
60,12743
70,5456
999,292


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq08_collision AS
SELECT
    c.*,

    CASE
        -- Speed limit
        WHEN c.speed_limit IS NOT NULL
             AND c.speed_limit NOT IN (-1, 20, 30, 40, 50, 60, 70)
        THEN 1

        -- Light conditions
        WHEN c.light_conditions IS NOT NULL
             AND c.light_conditions != -1
             AND c.light_conditions NOT IN (1, 4, 5, 6, 7)
        THEN 1

        -- Weather conditions
        WHEN c.weather_conditions IS NOT NULL
             AND c.weather_conditions != -1
             AND c.weather_conditions NOT IN (1, 2, 3, 4, 5, 6, 7, 8, 9)
        THEN 1

        -- Road surface conditions
        WHEN c.road_surface_conditions IS NOT NULL
             AND c.road_surface_conditions != -1
             AND c.road_surface_conditions NOT IN (1, 2, 3, 4, 5, 6, 7, 9)
        THEN 1

        -- Special conditions
        WHEN c.special_conditions_at_site IS NOT NULL
             AND c.special_conditions_at_site != -1
             AND c.special_conditions_at_site NOT IN (0, 1, 2, 3, 4, 5, 6, 7, 9)
        THEN 1

        -- Carriageway hazards
        WHEN c.carriageway_hazards IS NOT NULL
             AND c.carriageway_hazards != -1
             AND c.carriageway_hazards NOT IN
                 (0, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 99)
        THEN 1

        ELSE 0
    END AS dq08_collision_fail

FROM workspace.default.silver_collisions_candidate c;

In [0]:
%sql

SELECT
    dq08_collision_fail,
    COUNT(*) AS record_count
FROM dq08_collision
GROUP BY dq08_collision_fail
ORDER BY dq08_collision_fail;

dq08_collision_fail,record_count
0,100920
1,607


In [0]:
%sql

SELECT
    speed_limit,
    light_conditions,
    weather_conditions,
    road_surface_conditions,
    special_conditions_at_site,
    carriageway_hazards,
    COUNT(*) AS failure_count
FROM dq08_collision
WHERE dq08_collision_fail = 1
GROUP BY
    speed_limit,
    light_conditions,
    weather_conditions,
    road_surface_conditions,
    special_conditions_at_site,
    carriageway_hazards
ORDER BY failure_count DESC;

speed_limit,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards,failure_count
999,1,1,1,-1,0,93
999,1,1,1,0,0,52
30,1,88,1,-1,0,44
30,1,88,1,0,0,32
20,1,88,1,0,0,25
60,1,88,1,-1,0,19
999,4,1,1,-1,0,18
30,4,88,2,-1,0,17
999,4,1,1,0,0,15
30,4,88,1,-1,0,14


In [0]:
%sql

SELECT
    collision_index,
    speed_limit,
    light_conditions,
    weather_conditions,
    road_surface_conditions,
    special_conditions_at_site,
    carriageway_hazards
FROM dq08_collision
WHERE dq08_collision_fail = 1
LIMIT 20;

collision_index,speed_limit,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards
2024122400683,999,4,1,1,0,0
2024141495942,30,4,88,2,-1,0
2024010492142,30,4,88,1,0,0
2024221403313,999,1,1,1,-1,0
2024141432241,50,1,88,2,-1,0
2024544765124,30,1,88,1,0,0
2024430203130,60,7,88,1,0,0
2024031439234,60,6,88,2,-1,0
2024061533889,30,4,88,1,-1,0
2024061491715,999,1,1,1,-1,0


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq08_collision AS
SELECT
    c.*,

    CASE
        -- Speed limit
        WHEN c.speed_limit IS NOT NULL
             AND c.speed_limit NOT IN (-1, 20, 30, 40, 50, 60, 70)
        THEN 1

        -- Light conditions
        WHEN c.light_conditions IS NOT NULL
             AND c.light_conditions NOT IN (-1, 1, 4, 5, 6, 7)
        THEN 1

        -- Weather conditions
        WHEN c.weather_conditions IS NOT NULL
             AND c.weather_conditions NOT IN (-1, 1, 2, 3, 4, 5, 6, 7, 8, 9)
        THEN 1

        -- Road surface conditions
        WHEN c.road_surface_conditions IS NOT NULL
             AND c.road_surface_conditions NOT IN (-1, 1, 2, 3, 4, 5, 6, 7, 9)
        THEN 1

        -- Special conditions at site
        WHEN c.special_conditions_at_site IS NOT NULL
             AND c.special_conditions_at_site NOT IN (-1, 0, 1, 2, 3, 4, 5, 6, 7, 9)
        THEN 1

        -- Carriageway hazards
        WHEN c.carriageway_hazards IS NOT NULL
             AND c.carriageway_hazards NOT IN
                 (-1, 0, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 99)
        THEN 1

        ELSE 0
    END AS dq08_collision_fail

FROM workspace.default.silver_collisions_candidate c;

In [0]:
%sql

SELECT
    dq08_collision_fail,
    COUNT(*) AS record_count
FROM dq08_collision
GROUP BY dq08_collision_fail
ORDER BY dq08_collision_fail;

dq08_collision_fail,record_count
0,100920
1,607


In [0]:
%sql

SELECT
    SUM(CASE WHEN speed_limit NOT IN (-1,20,30,40,50,60,70) THEN 1 ELSE 0 END) AS bad_speed,
    SUM(CASE WHEN light_conditions NOT IN (-1,1,4,5,6,7) THEN 1 ELSE 0 END) AS bad_light,
    SUM(CASE WHEN weather_conditions NOT IN (-1,1,2,3,4,5,6,7,8,9) THEN 1 ELSE 0 END) AS bad_weather,
    SUM(CASE WHEN road_surface_conditions NOT IN (-1,1,2,3,4,5,6,7,9) THEN 1 ELSE 0 END) AS bad_surface,
    SUM(CASE WHEN special_conditions_at_site NOT IN (-1,0,1,2,3,4,5,6,7,9) THEN 1 ELSE 0 END) AS bad_special,
    SUM(CASE WHEN carriageway_hazards NOT IN (-1,0,11,12,13,14,15,16,17,18,19,20,21,99) THEN 1 ELSE 0 END) AS bad_hazard
FROM workspace.default.silver_collisions_candidate;

bad_speed,bad_light,bad_weather,bad_surface,bad_special,bad_hazard
292,0,315,0,0,0


In [0]:
%sql

DESCRIBE workspace.default.silver_vehicles_candidate;

col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
vehicle_reference,int,null
vehicle_type,int,null
towing_and_articulation,int,null
vehicle_manoeuvre_historic,int,null
vehicle_manoeuvre,int,null
vehicle_direction_from,int,null
vehicle_direction_to,int,null


In [0]:
%sql

SELECT
    age_of_driver,
    COUNT(*) AS record_count
FROM workspace.default.silver_vehicles_candidate
GROUP BY age_of_driver
ORDER BY age_of_driver;

age_of_driver,record_count
-50,359
-1,28569
1,1
2,3
3,5
4,11
5,17
6,32
7,39
8,60


In [0]:
%sql

SELECT
    age_of_vehicle,
    COUNT(*) AS record_count
FROM workspace.default.silver_vehicles_candidate
GROUP BY age_of_vehicle
ORDER BY age_of_vehicle;

age_of_vehicle,record_count
-1,50442
0,5433
1,9084
2,7205
3,6944
4,6459
5,8158
6,8047
7,8354
8,8569


In [0]:
%sql

SELECT
    age_of_driver,
    COUNT(*) AS record_count
FROM workspace.default.silver_vehicles_candidate
WHERE age_of_driver < -1
   OR age_of_driver > 99
GROUP BY age_of_driver
ORDER BY age_of_driver;

age_of_driver,record_count
-50,359


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq08_vehicle AS
SELECT
    v.*,

    CASE
        WHEN age_of_driver IS NOT NULL
             AND age_of_driver < -1
        THEN 1
        ELSE 0
    END AS dq08_vehicle_fail

FROM workspace.default.silver_vehicles_candidate v;

In [0]:
%sql

SELECT
    dq08_vehicle_fail,
    COUNT(*) AS record_count
FROM dq08_vehicle
GROUP BY dq08_vehicle_fail
ORDER BY dq08_vehicle_fail;

dq08_vehicle_fail,record_count
0,183155
1,359


In [0]:
%sql

DESCRIBE workspace.default.silver_casualties_candidate;

col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
vehicle_reference,int,null
casualty_reference,int,null
casualty_class,int,null
sex_of_casualty,int,null
age_of_casualty,int,null
age_band_of_casualty,int,null
casualty_severity,int,null


In [0]:
%sql

SELECT
    age_of_casualty,
    COUNT(*) AS record_count
FROM workspace.default.silver_casualties_candidate
GROUP BY age_of_casualty
ORDER BY age_of_casualty;

age_of_casualty,record_count
-1,2544
0,193
1,203
2,264
3,370
4,457
5,524
6,547
7,589
8,661


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq08_casualty AS
SELECT
    c.*,

    CASE
        WHEN age_of_casualty IS NOT NULL
             AND age_of_casualty != -1
             AND (
                 age_of_casualty < 0
                 OR age_of_casualty > 99
             )
        THEN 1
        ELSE 0
    END AS dq08_casualty_fail

FROM workspace.default.silver_casualties_candidate c;

In [0]:
%sql

SELECT
    dq08_casualty_fail,
    COUNT(*) AS record_count
FROM dq08_casualty
GROUP BY dq08_casualty_fail
ORDER BY dq08_casualty_fail;

dq08_casualty_fail,record_count
0,127908
1,364


In [0]:
%sql

SELECT
    age_of_casualty,
    COUNT(*) AS record_count
FROM dq08_casualty
WHERE dq08_casualty_fail = 1
GROUP BY age_of_casualty
ORDER BY age_of_casualty;

age_of_casualty,record_count
250,364


In [0]:
%sql

SHOW VIEWS;

namespace,viewName,isTemporary,isMaterialized,isMetric
default,roadsafe_week03_lineage_demo_view,false,false,false
,dq04_collision,true,false,false
,dq08_casualty,true,false,false
,dq08_collision,true,false,false
,dq08_vehicle,true,false,false


In [0]:
%sql

SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_casualties,false
default,bronze_collision_lookup,false
default,bronze_collisions,false
default,bronze_vehicles,false
default,roadsafe_week03_bronze_demo_collisions,false
default,roadsafe_week03_lineage_demo_view,false
default,silver_casualties_candidate,false
default,silver_collisions_candidate,false
default,silver_vehicles_candidate,false
,dq04_collision,true


In [0]:
%sql
DESCRIBE workspace.default.silver_collisions_candidate;

col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
location_easting_osgr,int,null
location_northing_osgr,int,null
longitude,double,null
latitude,double,null
police_force,int,null
collision_severity,int,null
number_of_vehicles,int,null


In [0]:
%sql
DESCRIBE workspace.default.silver_vehicles_candidate;

col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
vehicle_reference,int,null
vehicle_type,int,null
towing_and_articulation,int,null
vehicle_manoeuvre_historic,int,null
vehicle_manoeuvre,int,null
vehicle_direction_from,int,null
vehicle_direction_to,int,null


In [0]:
%sql
DESCRIBE workspace.default.silver_casualties_candidate;

col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
vehicle_reference,int,null
casualty_reference,int,null
casualty_class,int,null
sex_of_casualty,int,null
age_of_casualty,int,null
age_band_of_casualty,int,null
casualty_severity,int,null


In [0]:
%sql

DESCRIBE workspace.default.silver_casualties_candidate;

col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
vehicle_reference,int,null
casualty_reference,int,null
casualty_class,int,null
sex_of_casualty,int,null
age_of_casualty,int,null
age_band_of_casualty,int,null
casualty_severity,int,null


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq01_collision AS
SELECT
    c.*,
    CASE
        WHEN collision_index IS NULL
          OR TRIM(collision_index) = ''
        THEN 1
        WHEN COUNT(*) OVER (PARTITION BY collision_index) > 1
        THEN 1
        ELSE 0
    END AS dq01_collision_fail
FROM workspace.default.silver_collisions_candidate c;

In [0]:
%sql

SELECT
    dq01_collision_fail,
    COUNT(*) AS record_count
FROM dq01_collision
GROUP BY dq01_collision_fail
ORDER BY dq01_collision_fail;

dq01_collision_fail,record_count
0,100427
1,1100


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq02_vehicle AS
SELECT
    v.*,

    CASE
        WHEN v.collision_index IS NULL
             OR TRIM(v.collision_index) = ''
        THEN 1

        WHEN c.collision_index IS NULL
        THEN 1

        WHEN v.vehicle_reference IS NULL
        THEN 1

        ELSE 0
    END AS dq02_vehicle_fail

FROM workspace.default.silver_vehicles_candidate v
LEFT JOIN workspace.default.silver_collisions_candidate c
    ON v.collision_index = c.collision_index;

In [0]:
%sql

SELECT
    dq02_vehicle_fail,
    COUNT(*) AS record_count
FROM dq02_vehicle
GROUP BY dq02_vehicle_fail
ORDER BY dq02_vehicle_fail;

dq02_vehicle_fail,record_count
0,182928
1,1500


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq03_casualty AS
SELECT
    ca.*,

    CASE
        WHEN ca.collision_index IS NULL
             OR TRIM(ca.collision_index) = ''
        THEN 1

        WHEN c.collision_index IS NULL
        THEN 1

        WHEN ca.vehicle_reference IS NULL
        THEN 1

        WHEN v.vehicle_reference IS NULL
        THEN 1

        WHEN ca.casualty_reference IS NULL
        THEN 1

        ELSE 0
    END AS dq03_casualty_fail

FROM workspace.default.silver_casualties_candidate ca

LEFT JOIN workspace.default.silver_collisions_candidate c
    ON ca.collision_index = c.collision_index

LEFT JOIN workspace.default.silver_vehicles_candidate v
    ON ca.collision_index = v.collision_index
   AND ca.vehicle_reference = v.vehicle_reference;

In [0]:
%sql

SELECT
    dq03_casualty_fail,
    COUNT(*) AS record_count
FROM dq03_casualty
GROUP BY dq03_casualty_fail
ORDER BY dq03_casualty_fail;

dq03_casualty_fail,record_count
0,125776
1,3107


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq01_collision AS
SELECT
    c.*,

    CASE
        WHEN collision_index IS NULL
          OR TRIM(collision_index) = ''
        THEN 'FAIL'

        WHEN COUNT(*) OVER (PARTITION BY collision_index) > 1
        THEN 'FAIL'

        ELSE 'PASS'
    END AS dq01_status

FROM workspace.default.silver_collisions_candidate c;

In [0]:
%sql

SELECT
    dq01_status,
    COUNT(*) AS record_count
FROM dq01_collision
GROUP BY dq01_status
ORDER BY dq01_status;

dq01_status,record_count
FAIL,1100
PASS,100427


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq02_vehicle AS
SELECT
    v.*,

    CASE
        WHEN v.collision_index IS NULL
             OR TRIM(v.collision_index) = ''
        THEN 'FAIL'

        WHEN c.collision_index IS NULL
        THEN 'FAIL'

        WHEN v.vehicle_reference IS NULL
        THEN 'FAIL'

        ELSE 'PASS'
    END AS dq02_status

FROM workspace.default.silver_vehicles_candidate v
LEFT JOIN workspace.default.silver_collisions_candidate c
    ON v.collision_index = c.collision_index;

In [0]:
%sql

SELECT
    dq02_status,
    COUNT(*) AS record_count
FROM dq02_vehicle
GROUP BY dq02_status
ORDER BY dq02_status;

dq02_status,record_count
FAIL,1500
PASS,182928


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq03_casualty AS
SELECT
    ca.*,

    CASE
        WHEN ca.collision_index IS NULL
             OR TRIM(ca.collision_index) = ''
        THEN 'FAIL'

        WHEN c.collision_index IS NULL
        THEN 'FAIL'

        WHEN ca.vehicle_reference IS NULL
        THEN 'FAIL'

        WHEN v.vehicle_reference IS NULL
        THEN 'FAIL'

        WHEN ca.casualty_reference IS NULL
        THEN 'FAIL'

        ELSE 'PASS'
    END AS dq03_status

FROM workspace.default.silver_casualties_candidate ca

LEFT JOIN workspace.default.silver_collisions_candidate c
    ON ca.collision_index = c.collision_index

LEFT JOIN workspace.default.silver_vehicles_candidate v
    ON ca.collision_index = v.collision_index
   AND ca.vehicle_reference = v.vehicle_reference;

In [0]:
%sql

SELECT
    dq03_status,
    COUNT(*) AS record_count
FROM dq03_casualty
GROUP BY dq03_status
ORDER BY dq03_status;

dq03_status,record_count
FAIL,3107
PASS,125776


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq04_collision AS
SELECT
    c.*,

    CASE
        WHEN speed_limit IS NOT NULL
             AND speed_limit NOT IN (-1, 20, 30, 40, 50, 60, 70)
        THEN 'FAIL'

        WHEN light_conditions IS NOT NULL
             AND light_conditions NOT IN (-1, 1, 4, 5, 6, 7)
        THEN 'FAIL'

        WHEN weather_conditions IS NOT NULL
             AND weather_conditions NOT IN (-1, 1, 2, 3, 4, 5, 6, 7, 8, 9)
        THEN 'FAIL'

        WHEN road_surface_conditions IS NOT NULL
             AND road_surface_conditions NOT IN (-1, 1, 2, 3, 4, 5, 6, 7, 9)
        THEN 'FAIL'

        WHEN special_conditions_at_site IS NOT NULL
             AND special_conditions_at_site NOT IN
                 (-1, 0, 1, 2, 3, 4, 5, 6, 7, 9)
        THEN 'FAIL'

        WHEN carriageway_hazards IS NOT NULL
             AND carriageway_hazards NOT IN
                 (-1, 0, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 99)
        THEN 'FAIL'

        ELSE 'PASS'
    END AS dq04_status

FROM workspace.default.silver_collisions_candidate c;

In [0]:
%sql

SELECT
    dq04_status,
    COUNT(*) AS record_count
FROM dq04_collision
GROUP BY dq04_status
ORDER BY dq04_status;

dq04_status,record_count
FAIL,607
PASS,100920


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq05_collision AS
SELECT
    c.*,

    CASE
        WHEN date IS NULL
             OR TRIM(date) = ''
        THEN 'FAIL'

        WHEN try_to_date(date, 'dd/MM/yyyy') IS NULL
        THEN 'FAIL'

        WHEN time IS NULL
             OR TRIM(time) = ''
        THEN 'FAIL'

        WHEN try_to_timestamp(
                 CONCAT(date, ' ', time),
                 'dd/MM/yyyy HH:mm'
             ) IS NULL
        THEN 'FAIL'

        WHEN YEAR(
                 try_to_timestamp(
                     CONCAT(date, ' ', time),
                     'dd/MM/yyyy HH:mm'
                 )
             ) <> 2024
        THEN 'FAIL'

        ELSE 'PASS'
    END AS dq05_status

FROM workspace.default.silver_collisions_candidate c;

In [0]:
%sql

SELECT
    dq05_status,
    COUNT(*) AS record_count
FROM dq05_collision
GROUP BY dq05_status
ORDER BY dq05_status;

dq05_status,record_count
FAIL,501
PASS,101026


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq06_collision AS
SELECT
    c.*,

    CASE
        WHEN latitude IS NOT NULL
             AND (latitude < 49.0 OR latitude > 61.0)
        THEN 'FAIL'

        WHEN longitude IS NOT NULL
             AND (longitude < -11.0 OR longitude > 3.0)
        THEN 'FAIL'

        ELSE 'PASS'
    END AS dq06_status

FROM workspace.default.silver_collisions_candidate c;

In [0]:
%sql

SELECT
    dq06_status,
    COUNT(*) AS record_count
FROM dq06_collision
GROUP BY dq06_status
ORDER BY dq06_status;

dq06_status,record_count
FAIL,388
PASS,101139


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq07_casualty AS
SELECT
    ca.*,

    CASE
        WHEN age_of_casualty IS NOT NULL
             AND (
                 age_of_casualty < -1
                 OR age_of_casualty > 120
             )
        THEN 'FAIL'

        ELSE 'PASS'
    END AS dq07_status

FROM workspace.default.silver_casualties_candidate ca;

In [0]:
%sql

SELECT
    dq07_status,
    COUNT(*) AS record_count
FROM dq07_casualty
GROUP BY dq07_status
ORDER BY dq07_status;

dq07_status,record_count
FAIL,364
PASS,127908


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq08_collision AS
SELECT
    c.*,

    CASE
        WHEN speed_limit NOT IN (-1, 20, 30, 40, 50, 60, 70, 999)
          OR light_conditions NOT IN (-1, 1, 4, 5, 6, 7)
          OR weather_conditions NOT IN (-1, 1, 2, 3, 4, 5, 6, 7, 8, 9)
          OR road_surface_conditions NOT IN (-1, 1, 2, 3, 4, 5, 6, 7, 9)
          OR special_conditions_at_site NOT IN (-1, 0, 1, 2, 3, 4, 5, 6, 7, 9)
          OR carriageway_hazards NOT IN (-1, 0, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 99)
        THEN 'FAIL'
        ELSE 'PASS'
    END AS dq08_status

FROM workspace.default.silver_collisions_candidate c;

In [0]:
%sql

SELECT
    dq08_status,
    COUNT(*) AS record_count
FROM dq08_collision
GROUP BY dq08_status
ORDER BY dq08_status;

dq08_status,record_count
FAIL,315
PASS,101212


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW collision_dq_final AS
SELECT
    c.*,

    d1.dq01_status,

    d4.dq04_status,
    d5.dq05_status,
    d6.dq06_status,
    d8.dq08_status,

    CASE
        WHEN d1.dq01_status = 'FAIL'
          OR d4.dq04_status = 'FAIL'
          OR d5.dq05_status = 'FAIL'
          OR d6.dq06_status = 'FAIL'
          OR d8.dq08_status = 'FAIL'
        THEN 'FAIL'
        ELSE 'PASS'
    END AS overall_status

FROM workspace.default.silver_collisions_candidate c

LEFT JOIN (
    SELECT
        collision_index,
        MAX(CASE WHEN dq01_status = 'FAIL' THEN 'FAIL' ELSE 'PASS' END) AS dq01_status
    FROM dq01_collision
    GROUP BY collision_index
) d1
    ON c.collision_index = d1.collision_index

LEFT JOIN (
    SELECT
        collision_index,
        MAX(CASE WHEN dq04_status = 'FAIL' THEN 'FAIL' ELSE 'PASS' END) AS dq04_status
    FROM dq04_collision
    GROUP BY collision_index
) d4
    ON c.collision_index = d4.collision_index

LEFT JOIN (
    SELECT
        collision_index,
        MAX(CASE WHEN dq05_status = 'FAIL' THEN 'FAIL' ELSE 'PASS' END) AS dq05_status
    FROM dq05_collision
    GROUP BY collision_index
) d5
    ON c.collision_index = d5.collision_index

LEFT JOIN (
    SELECT
        collision_index,
        MAX(CASE WHEN dq06_status = 'FAIL' THEN 'FAIL' ELSE 'PASS' END) AS dq06_status
    FROM dq06_collision
    GROUP BY collision_index
) d6
    ON c.collision_index = d6.collision_index

LEFT JOIN (
    SELECT
        collision_index,
        MAX(CASE WHEN dq08_status = 'FAIL' THEN 'FAIL' ELSE 'PASS' END) AS dq08_status
    FROM dq08_collision
    GROUP BY collision_index
) d8
    ON c.collision_index = d8.collision_index;

In [0]:
%sql

SELECT
    overall_status,
    COUNT(*) AS record_count
FROM collision_dq_final
GROUP BY overall_status
ORDER BY overall_status;

overall_status,record_count
FAIL,2480
PASS,99047


In [0]:
%sql

DROP VIEW IF EXISTS collision_dq_final;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq08_vehicle AS
SELECT
    v.*,

    CASE
        WHEN age_of_driver IS NOT NULL
             AND (age_of_driver < -1 OR age_of_driver > 120)
        THEN 'FAIL'
        ELSE 'PASS'
    END AS dq08_status

FROM workspace.default.silver_vehicles_candidate v;

In [0]:
%sql

SELECT
    dq08_status,
    COUNT(*) AS record_count
FROM dq08_vehicle
GROUP BY dq08_status
ORDER BY dq08_status;

dq08_status,record_count
FAIL,359
PASS,183155


In [0]:
%sql

SELECT COUNT(*) AS candidate_count
FROM workspace.default.silver_vehicles_candidate;

candidate_count
183514


In [0]:
%sql

SELECT COUNT(*) AS dq08_count
FROM dq08_vehicle;

dq08_count
183514


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW vehicle_dq_final AS
SELECT
    v.*,

    COALESCE(d2.dq02_status, 'FAIL') AS dq02_status,
    COALESCE(d8.dq08_status, 'PASS') AS dq08_status,

    CASE
        WHEN COALESCE(d2.dq02_status, 'FAIL') = 'FAIL'
          OR COALESCE(d8.dq08_status, 'PASS') = 'FAIL'
        THEN 'FAIL'
        ELSE 'PASS'
    END AS overall_status

FROM workspace.default.silver_vehicles_candidate v

LEFT JOIN (
    SELECT
        collision_index,
        vehicle_reference,
        MAX(
            CASE
                WHEN dq02_status = 'FAIL' THEN 'FAIL'
                ELSE 'PASS'
            END
        ) AS dq02_status
    FROM dq02_vehicle
    GROUP BY collision_index, vehicle_reference
) d2
    ON v.collision_index = d2.collision_index
   AND v.vehicle_reference = d2.vehicle_reference

LEFT JOIN (
    SELECT
        collision_index,
        vehicle_reference,
        MAX(
            CASE
                WHEN dq08_status = 'FAIL' THEN 'FAIL'
                ELSE 'PASS'
            END
        ) AS dq08_status
    FROM dq08_vehicle
    GROUP BY collision_index, vehicle_reference
) d8
    ON v.collision_index = d8.collision_index
   AND v.vehicle_reference = d8.vehicle_reference;

In [0]:
%sql

SELECT
    overall_status,
    COUNT(*) AS record_count
FROM vehicle_dq_final
GROUP BY overall_status
ORDER BY overall_status;

overall_status,record_count
FAIL,1859
PASS,181655


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW dq08_casualty AS
SELECT
    ca.*,

    CASE
        WHEN age_of_casualty IS NOT NULL
             AND (age_of_casualty < -1 OR age_of_casualty > 120)
        THEN 'FAIL'
        ELSE 'PASS'
    END AS dq08_status

FROM workspace.default.silver_casualties_candidate ca;

In [0]:
%sql

SELECT
    dq08_status,
    COUNT(*) AS record_count
FROM dq08_casualty
GROUP BY dq08_status
ORDER BY dq08_status;

dq08_status,record_count
FAIL,364
PASS,127908


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW casualty_dq_final AS
SELECT
    ca.*,

    COALESCE(d3.dq03_status, 'FAIL') AS dq03_status,
    COALESCE(d7.dq07_status, 'PASS') AS dq07_status,
    COALESCE(d8.dq08_status, 'PASS') AS dq08_status,

    CASE
        WHEN COALESCE(d3.dq03_status, 'FAIL') = 'FAIL'
          OR COALESCE(d7.dq07_status, 'PASS') = 'FAIL'
          OR COALESCE(d8.dq08_status, 'PASS') = 'FAIL'
        THEN 'FAIL'
        ELSE 'PASS'
    END AS overall_status

FROM workspace.default.silver_casualties_candidate ca

LEFT JOIN (
    SELECT
        collision_index,
        vehicle_reference,
        casualty_reference,
        MAX(
            CASE
                WHEN dq03_status = 'FAIL' THEN 'FAIL'
                ELSE 'PASS'
            END
        ) AS dq03_status
    FROM dq03_casualty
    GROUP BY
        collision_index,
        vehicle_reference,
        casualty_reference
) d3
    ON ca.collision_index = d3.collision_index
   AND ca.vehicle_reference = d3.vehicle_reference
   AND ca.casualty_reference = d3.casualty_reference

LEFT JOIN (
    SELECT
        collision_index,
        vehicle_reference,
        casualty_reference,
        MAX(
            CASE
                WHEN dq07_status = 'FAIL' THEN 'FAIL'
                ELSE 'PASS'
            END
        ) AS dq07_status
    FROM dq07_casualty
    GROUP BY
        collision_index,
        vehicle_reference,
        casualty_reference
) d7
    ON ca.collision_index = d7.collision_index
   AND ca.vehicle_reference = d7.vehicle_reference
   AND ca.casualty_reference = d7.casualty_reference

LEFT JOIN (
    SELECT
        collision_index,
        vehicle_reference,
        casualty_reference,
        MAX(
            CASE
                WHEN dq08_status = 'FAIL' THEN 'FAIL'
                ELSE 'PASS'
            END
        ) AS dq08_status
    FROM dq08_casualty
    GROUP BY
        collision_index,
        vehicle_reference,
        casualty_reference
) d8
    ON ca.collision_index = d8.collision_index
   AND ca.vehicle_reference = d8.vehicle_reference
   AND ca.casualty_reference = d8.casualty_reference;

In [0]:
%sql

SELECT
    overall_status,
    COUNT(*) AS record_count
FROM casualty_dq_final
GROUP BY overall_status
ORDER BY overall_status;

overall_status,record_count
FAIL,3455
PASS,124817


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.quarantine_collisions AS
SELECT *
FROM collision_dq_final
WHERE overall_status = 'FAIL';

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS quarantine_collision_count
FROM workspace.default.quarantine_collisions;

quarantine_collision_count
2480


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.quarantine_vehicles AS
SELECT *
FROM vehicle_dq_final
WHERE overall_status = 'FAIL';

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS quarantine_vehicle_count
FROM workspace.default.quarantine_vehicles;

quarantine_vehicle_count
1859


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.quarantine_casualties AS
SELECT *
FROM casualty_dq_final
WHERE overall_status = 'FAIL';

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS quarantine_casualty_count
FROM workspace.default.quarantine_casualties;

quarantine_casualty_count
3455


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.trusted_silver_collisions AS
SELECT *
FROM collision_dq_final
WHERE overall_status = 'PASS';

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS trusted_collision_count
FROM workspace.default.trusted_silver_collisions;

trusted_collision_count
99047


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.trusted_silver_vehicles AS
SELECT *
FROM vehicle_dq_final
WHERE overall_status = 'PASS';

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS trusted_vehicle_count
FROM workspace.default.trusted_silver_vehicles;

trusted_vehicle_count
181655


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.trusted_silver_casualties AS
SELECT *
FROM casualty_dq_final
WHERE overall_status = 'PASS';


num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS trusted_casualty_count
FROM workspace.default.trusted_silver_casualties;

trusted_casualty_count
124817


In [0]:
%sql

SELECT
    'Collision' AS entity,
    (SELECT COUNT(*) FROM workspace.default.silver_collisions_candidate) AS candidate_count,
    (SELECT COUNT(*) FROM workspace.default.trusted_silver_collisions) AS trusted_count,
    (SELECT COUNT(*) FROM workspace.default.quarantine_collisions) AS quarantine_count,
    (SELECT COUNT(*) FROM workspace.default.trusted_silver_collisions)
      + (SELECT COUNT(*) FROM workspace.default.quarantine_collisions) AS routed_count

UNION ALL

SELECT
    'Vehicle',
    (SELECT COUNT(*) FROM workspace.default.silver_vehicles_candidate),
    (SELECT COUNT(*) FROM workspace.default.trusted_silver_vehicles),
    (SELECT COUNT(*) FROM workspace.default.quarantine_vehicles),
    (SELECT COUNT(*) FROM workspace.default.trusted_silver_vehicles)
      + (SELECT COUNT(*) FROM workspace.default.quarantine_vehicles)

UNION ALL

SELECT
    'Casualty',
    (SELECT COUNT(*) FROM workspace.default.silver_casualties_candidate),
    (SELECT COUNT(*) FROM workspace.default.trusted_silver_casualties),
    (SELECT COUNT(*) FROM workspace.default.quarantine_casualties),
    (SELECT COUNT(*) FROM workspace.default.trusted_silver_casualties)
      + (SELECT COUNT(*) FROM workspace.default.quarantine_casualties);

entity,candidate_count,trusted_count,quarantine_count,routed_count
Collision,101527,99047,2480,101527
Vehicle,183514,181655,1859,183514
Casualty,128272,124817,3455,128272


In [0]:
%sql

SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_casualties,false
default,bronze_collision_lookup,false
default,bronze_collisions,false
default,bronze_vehicles,false
default,quarantine_casualties,false
default,quarantine_collisions,false
default,quarantine_vehicles,false
default,roadsafe_week03_bronze_demo_collisions,false
default,roadsafe_week03_lineage_demo_view,false
default,silver_casualties_candidate,false


In [0]:
%sql

SELECT
    'Collision Trusted' AS table_name,
    COUNT(*) AS record_count
FROM workspace.default.trusted_silver_collisions

UNION ALL

SELECT
    'Vehicle Trusted',
    COUNT(*)
FROM workspace.default.trusted_silver_vehicles

UNION ALL

SELECT
    'Casualty Trusted',
    COUNT(*)
FROM workspace.default.trusted_silver_casualties

UNION ALL

SELECT
    'Collision Quarantine',
    COUNT(*)
FROM workspace.default.quarantine_collisions

UNION ALL

SELECT
    'Vehicle Quarantine',
    COUNT(*)
FROM workspace.default.quarantine_vehicles

UNION ALL

SELECT
    'Casualty Quarantine',
    COUNT(*)
FROM workspace.default.quarantine_casualties;

table_name,record_count
Collision Trusted,99047
Vehicle Trusted,181655
Casualty Trusted,124817
Collision Quarantine,2480
Vehicle Quarantine,1859
Casualty Quarantine,3455


In [0]:
# WEEK 7 - Inspect available catalogs, schemas and tables

print("Databricks Spark session is working.")

print("\nCatalogs:")
display(spark.sql("SHOW CATALOGS"))

Databricks Spark session is working.

Catalogs:


catalog
road_safety
samples
system
workspace


In [0]:
# Show schemas in the current catalog

current_catalog = spark.sql("SELECT current_catalog()").first()[0]

print("Current catalog:", current_catalog)

display(
    spark.sql(f"SHOW SCHEMAS IN `{current_catalog}`")
)

Current catalog: workspace


databaseName
default
information_schema


In [0]:
# Show tables in the current schema

current_schema = spark.sql("SELECT current_schema()").first()[0]

print("Current schema:", current_schema)

display(
    spark.sql(f"SHOW TABLES IN `{current_catalog}`.`{current_schema}`")
)

Current schema: default


database,tableName,isTemporary
default,bronze_casualties,false
default,bronze_collision_lookup,false
default,bronze_collisions,false
default,bronze_vehicles,false
default,quarantine_casualties,false
default,quarantine_collisions,false
default,quarantine_vehicles,false
default,roadsafe_week03_bronze_demo_collisions,false
default,roadsafe_week03_lineage_demo_view,false
default,silver_casualties_candidate,false


In [0]:
# Search for tables whose names contain "trusted" or "silver"

tables_df = spark.sql(
    f"SHOW TABLES IN `{current_catalog}`.`{current_schema}`"
)

display(
    tables_df.filter(
        "lower(tableName) LIKE '%trusted%' OR lower(tableName) LIKE '%silver%'"
    )
)

database,tableName,isTemporary
default,silver_casualties_candidate,false
default,silver_collisions_candidate,false
default,silver_vehicles_candidate,false
default,trusted_silver_casualties,false
default,trusted_silver_collisions,false
default,trusted_silver_vehicles,false


In [0]:
# WEEK 7 - Step 1: Inspect Trusted Silver tables

tables = [
    "trusted_silver_casualties",
    "trusted_silver_collisions",
    "trusted_silver_vehicles"
]

for table in tables:
    print("\n" + "=" * 70)
    print("TABLE:", table)
    print("=" * 70)
    
    display(spark.sql(f"DESCRIBE `{table}`"))


TABLE: trusted_silver_casualties


col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
vehicle_reference,int,null
casualty_reference,int,null
casualty_class,int,null
sex_of_casualty,int,null
age_of_casualty,int,null
age_band_of_casualty,int,null
casualty_severity,int,null



TABLE: trusted_silver_collisions


col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
location_easting_osgr,int,null
location_northing_osgr,int,null
longitude,double,null
latitude,double,null
police_force,int,null
collision_severity,int,null
number_of_vehicles,int,null



TABLE: trusted_silver_vehicles


col_name,data_type,comment
collision_index,string,null
collision_year,int,null
collision_ref_no,string,null
vehicle_reference,int,null
vehicle_type,int,null
towing_and_articulation,int,null
vehicle_manoeuvre_historic,int,null
vehicle_manoeuvre,int,null
vehicle_direction_from,int,null
vehicle_direction_to,int,null


In [0]:
# WEEK 7 - Step 2: Row counts

for table in tables:
    count = spark.sql(
        f"SELECT COUNT(*) AS row_count FROM `{table}`"
    ).first()["row_count"]
    
    print(f"{table}: {count:,} rows")

trusted_silver_casualties: 124,817 rows
trusted_silver_collisions: 99,047 rows
trusted_silver_vehicles: 181,655 rows


In [0]:
# WEEK 7 - Step 3: Sample records

for table in tables:
    print("\n" + "=" * 70)
    print("SAMPLE:", table)
    print("=" * 70)
    
    display(
        spark.sql(f"""
            SELECT *
            FROM `{table}`
            LIMIT 10
        """)
    )


SAMPLE: trusted_silver_casualties


collision_index,collision_year,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding,ingested_at,source_file,casualty_key,casualty_age_group,vulnerable_road_user_flag,severe_casualty_flag,dq03_status,dq07_status,dq08_status,overall_status
2024991534042,2024,991534042,1,1,3,2,39,7,3,1,3,0,0,-1,0,-1,-1,3,1,0,1,-1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024991534042_1,31-50,1,0,PASS,PASS,PASS,PASS
2024430342145,2024,430342145,2,1,1,2,25,5,3,0,0,0,0,0,1,4,E01028532,-1,0,null,null,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024430342145_1,18-30,0,0,PASS,PASS,PASS,PASS
2024471442093,2024,471442093,2,1,1,1,49,8,3,0,0,0,0,0,1,8,E01031491,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024471442093_1,31-50,0,0,PASS,PASS,PASS,PASS
2024101511084,2024,101511084,1,1,1,2,39,7,3,0,0,0,0,0,9,1,E01008859,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024101511084_1,31-50,0,0,PASS,PASS,PASS,PASS
2024161531062,2024,161531062,1,1,1,1,59,9,3,0,0,0,0,0,1,5,E01012902,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024161531062_1,51-70,0,0,PASS,PASS,PASS,PASS
2024421434943,2024,421434943,2,1,1,1,62,9,2,0,0,0,0,0,3,5,E01034081,5,1,1,0,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024421434943_1,51-70,0,1,PASS,PASS,PASS,PASS
2024010533954,2024,010533954,1,1,1,1,44,7,3,0,0,0,0,0,5,5,E01021790,3,1,0,1,3,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024010533954_1,31-50,0,0,PASS,PASS,PASS,PASS
2024131453448,2024,131453448,1,2,2,2,35,6,2,0,0,1,0,0,9,4,E01011447,7,1,1,0,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024131453448_2,31-50,0,1,PASS,PASS,PASS,PASS
2024010511909,2024,010511909,2,1,1,1,34,6,3,0,0,0,0,0,3,5,E01032741,3,1,0,1,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024010511909_1,31-50,0,0,PASS,PASS,PASS,PASS
2024070726166,2024,070726166,2,1,1,1,58,9,2,0,0,0,0,0,3,8,E01014131,7,1,1,0,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024070726166_1,51-70,0,1,PASS,PASS,PASS,PASS



SAMPLE: trusted_silver_collisions


collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,local_authority_district,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail_historic,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control_historic,pedestrian_crossing_physical_facilities_historic,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,ingested_at,source_file,collision_timestamp,collision_month,collision_hour,collision_weekday,weekend_flag,time_band,severe_collision_flag,urban_rural_group,dq01_status,dq04_status,dq05_status,dq06_status,dq08_status,overall_status
202417M119024,2024,17M119024,450057,519938,-1.22722,54.57219,17,3,2,1,05/12/2024,5,16:10,-1,E06000002,E06000002,E06000002,6,0,6,30,-1,0,-1,-1,-1,-1,-1,0,4,2,2,-1,-1,0,1,3,2,E01035190,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-12-05T16:10:00.000Z,12,16,5,0,12:00-17:59,0,Urban,PASS,PASS,PASS,PASS,PASS,PASS
202417S312124,2024,17S312124,445858,516830,-1.29265,54.54466,17,3,1,1,22/10/2024,3,14:56,-1,E06000004,E06000004,E06000004,6,0,6,30,-1,0,-1,-1,-1,-1,-1,0,1,1,1,-1,-1,0,1,3,2,E01012239,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-10-22T14:56:00.000Z,10,14,3,0,12:00-17:59,0,Urban,PASS,PASS,PASS,PASS,PASS,PASS
2024070110901,2024,070110901,364765,390979,-2.53157,53.41443,7,3,2,1,24/01/2024,4,17:50,-1,E06000007,E06000007,E06000007,6,0,6,30,3,13,4,6,0,0,0,0,4,1,1,0,0,0,2,2,2,E01012459,-1,0,null,null,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-01-24T17:50:00.000Z,1,17,4,0,12:00-17:59,0,Rural,PASS,PASS,PASS,PASS,PASS,PASS
2024041446676,2024,041446676,365989,419804,-2.5163,53.67359,4,2,1,1,22/05/2024,4,17:45,-1,E06000008,E06000008,E06000008,3,675,6,50,-1,0,-1,-1,-1,-1,-1,0,1,5,5,-1,-1,0,2,1,2,E01012628,6,1,1,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-22T17:45:00.000Z,5,17,4,0,12:00-17:59,1,Rural,PASS,PASS,PASS,PASS,PASS,PASS
2024041478641,2024,041478641,366864,430071,-2.50416,53.76592,4,2,2,1,11/08/2024,1,15:36,-1,E06000008,E06000008,E06000008,3,6119,3,50,-1,16,4,6,0,-1,-1,0,1,1,1,-1,-1,0,1,1,2,E01012581,7,1,1,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-08-11T15:36:00.000Z,8,15,1,1,12:00-17:59,1,Urban,PASS,PASS,PASS,PASS,PASS,PASS
2024161425653,2024,161425653,526509,404543,-0.09323,53.52234,16,3,2,2,19/03/2024,3,14:40,-1,E06000012,E06000012,E06000012,4,1203,6,40,-1,0,-1,6,0,-1,-1,0,1,9,1,-1,-1,0,2,2,2,E01013215,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-03-19T14:40:00.000Z,3,14,3,0,12:00-17:59,0,Rural,PASS,PASS,PASS,PASS,PASS,PASS
2024161438773,2024,161438773,488486,409919,-0.66503,53.57841,16,3,1,1,03/05/2024,6,08:19,-1,E06000013,E06000013,E06000013,6,0,6,30,-1,0,-1,6,0,-1,-1,0,1,1,1,-1,-1,0,1,1,2,E01013321,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-03T08:19:00.000Z,5,8,6,0,06:00-11:59,0,Urban,PASS,PASS,PASS,PASS,PASS,PASS
2024122400772,2024,122400772,458988,455182,-1.10184,53.98933,12,3,4,1,30/05/2024,5,08:29,-1,E06000014,E06000014,E06000014,6,0,6,30,0,0,-1,0,-1,0,0,0,1,1,1,0,0,0,1,1,2,E01013431,-1,0,null,null,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-30T08:29:00.000Z,5,8,5,0,06:00-11:59,0,Urban,PASS,PASS,PASS,P


SAMPLE: trusted_silver_vehicles


collision_index,collision_year,collision_ref_no,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre_historic,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane_historic,vehicle_location_restricted_lane,junction_location,skidding_and_overturning,hit_object_in_carriageway,vehicle_leaving_carriageway,hit_object_off_carriageway,first_point_of_impact,vehicle_left_hand_drive,journey_purpose_of_driver_historic,journey_purpose_of_driver,sex_of_driver,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding,ingested_at,source_file,vehicle_key,dq02_status,dq08_status,overall_status
2024471526976,2024,471526976,2,4,0,-1,19,4,8,-1,0,8,0,0,0,0,3,1,-1,9,1,23,5,234,1,16,MAKE AND MODEL REDACTED,8,E01020954,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471526976_2,PASS,PASS,PASS
2024471493223,2024,471493223,2,9,0,-1,2,0,0,-1,0,0,0,0,0,0,1,1,-1,6,-1,-1,-1,2198,1,15,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471493223_2,PASS,PASS,PASS
2024010504128,2024,010504128,2,9,0,2,2,0,0,0,0,0,0,0,0,0,2,1,6,6,3,-1,-1,2198,1,18,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024010504128_2,PASS,PASS,PASS
2024302400896,2024,302400896,2,9,0,18,19,1,4,0,0,0,0,0,0,0,1,1,6,6,1,56,9,6496,1,0,MAKE AND MODEL REDACTED,5,E01019782,0,2,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024302400896_2,PASS,PASS,PASS
2024631472488,2024,631472488,2,90,0,-1,19,7,3,-1,0,8,0,0,0,0,4,1,-1,9,1,55,8,2184,2,0,MAKE AND MODEL REDACTED,7,W01000623,0,4,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024631472488_2,PASS,PASS,PASS
2024471512096,2024,471512096,1,4,0,-1,19,2,6,-1,0,0,2,0,0,0,4,1,-1,9,1,57,9,449,1,0,MAKE AND MODEL REDACTED,5,E01021074,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471512096_1,PASS,PASS,PASS
2024211495435,2024,211495435,1,2,0,-1,3,3,3,-1,0,1,5,0,0,0,2,1,-1,6,2,17,4,49,1,21,MAKE AND MODEL REDACTED,3,E01029752,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024211495435_1,PASS,PASS,PASS
2024131459645,2024,131459645,1,9,0,-1,10,1,5,-1,0,8,0,0,0,0,3,1,-1,9,1,44,7,1591,1,5,HYUNDAI TUCSON,7,E01011942,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024131459645_1,PASS,PASS,PASS
2024401457067,2024,401457067,1,9,0,-1,6,7,3,-1,0,0,0,0,0,0,1,1,-1,6,1,26,6,1685,2,9,HYUNDAI TUCSON,10,E01017611,0,3,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024401457067_1,PASS,PASS,PASS
2024031497068,2024,031497068,1,9,0,-1,19,1,5,-1,0,0,5,0,1,4,1,1,-1,9,1,48,8,1598,1,3,HYUNDAI TUCSON,1,-1,0,5,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024031497068_1,PASS,PASS,PASS


In [0]:
# WEEK 7 - Step 4: Display exact Trusted Silver schemas

tables = [
    "trusted_silver_casualties",
    "trusted_silver_collisions",
    "trusted_silver_vehicles"
]

for table in tables:
    print("\n" + "=" * 80)
    print(f"SCHEMA: {table}")
    print("=" * 80)

    schema = spark.table(table).schema

    for field in schema.fields:
        print(f"{field.name:40} {field.dataType}")


SCHEMA: trusted_silver_casualties
collision_index                          StringType()
collision_year                           IntegerType()
collision_ref_no                         StringType()
vehicle_reference                        IntegerType()
casualty_reference                       IntegerType()
casualty_class                           IntegerType()
sex_of_casualty                          IntegerType()
age_of_casualty                          IntegerType()
age_band_of_casualty                     IntegerType()
casualty_severity                        IntegerType()
pedestrian_location                      IntegerType()
pedestrian_movement                      IntegerType()
car_passenger                            IntegerType()
bus_or_coach_passenger                   IntegerType()
pedestrian_road_maintenance_worker       IntegerType()
casualty_type                            IntegerType()
casualty_imd_decile                      IntegerType()
lsoa_of_casualty                

In [0]:
# WEEK 7 - Step 5: Preview important records

display(
    spark.sql("""
        SELECT *
        FROM trusted_silver_collisions
        LIMIT 20
    """)
)

collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,local_authority_district,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail_historic,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control_historic,pedestrian_crossing_physical_facilities_historic,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,ingested_at,source_file,collision_timestamp,collision_month,collision_hour,collision_weekday,weekend_flag,time_band,severe_collision_flag,urban_rural_group,dq01_status,dq04_status,dq05_status,dq06_status,dq08_status,overall_status
202417M119024,2024,17M119024,450057,519938,-1.22722,54.57219,17,3,2,1,05/12/2024,5,16:10,-1,E06000002,E06000002,E06000002,6,0,6,30,-1,0,-1,-1,-1,-1,-1,0,4,2,2,-1,-1,0,1,3,2,E01035190,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-12-05T16:10:00.000Z,12,16,5,0,12:00-17:59,0,Urban,PASS,PASS,PASS,PASS,PASS,PASS
202417S312124,2024,17S312124,445858,516830,-1.29265,54.54466,17,3,1,1,22/10/2024,3,14:56,-1,E06000004,E06000004,E06000004,6,0,6,30,-1,0,-1,-1,-1,-1,-1,0,1,1,1,-1,-1,0,1,3,2,E01012239,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-10-22T14:56:00.000Z,10,14,3,0,12:00-17:59,0,Urban,PASS,PASS,PASS,PASS,PASS,PASS
2024070110901,2024,070110901,364765,390979,-2.53157,53.41443,7,3,2,1,24/01/2024,4,17:50,-1,E06000007,E06000007,E06000007,6,0,6,30,3,13,4,6,0,0,0,0,4,1,1,0,0,0,2,2,2,E01012459,-1,0,null,null,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-01-24T17:50:00.000Z,1,17,4,0,12:00-17:59,0,Rural,PASS,PASS,PASS,PASS,PASS,PASS
2024041446676,2024,041446676,365989,419804,-2.5163,53.67359,4,2,1,1,22/05/2024,4,17:45,-1,E06000008,E06000008,E06000008,3,675,6,50,-1,0,-1,-1,-1,-1,-1,0,1,5,5,-1,-1,0,2,1,2,E01012628,6,1,1,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-22T17:45:00.000Z,5,17,4,0,12:00-17:59,1,Rural,PASS,PASS,PASS,PASS,PASS,PASS
2024041478641,2024,041478641,366864,430071,-2.50416,53.76592,4,2,2,1,11/08/2024,1,15:36,-1,E06000008,E06000008,E06000008,3,6119,3,50,-1,16,4,6,0,-1,-1,0,1,1,1,-1,-1,0,1,1,2,E01012581,7,1,1,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-08-11T15:36:00.000Z,8,15,1,1,12:00-17:59,1,Urban,PASS,PASS,PASS,PASS,PASS,PASS
2024161425653,2024,161425653,526509,404543,-0.09323,53.52234,16,3,2,2,19/03/2024,3,14:40,-1,E06000012,E06000012,E06000012,4,1203,6,40,-1,0,-1,6,0,-1,-1,0,1,9,1,-1,-1,0,2,2,2,E01013215,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-03-19T14:40:00.000Z,3,14,3,0,12:00-17:59,0,Rural,PASS,PASS,PASS,PASS,PASS,PASS
2024161438773,2024,161438773,488486,409919,-0.66503,53.57841,16,3,1,1,03/05/2024,6,08:19,-1,E06000013,E06000013,E06000013,6,0,6,30,-1,0,-1,6,0,-1,-1,0,1,1,1,-1,-1,0,1,1,2,E01013321,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-03T08:19:00.000Z,5,8,6,0,06:00-11:59,0,Urban,PASS,PASS,PASS,PASS,PASS,PASS
2024122400772,2024,122400772,458988,455182,-1.10184,53.98933,12,3,4,1,30/05/2024,5,08:29,-1,E06000014,E06000014,E06000014,6,0,6,30,0,0,-1,0,-1,0,0,0,1,1,1,0,0,0,1,1,2,E01013431,-1,0,null,null,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-05-30T08:29:00.000Z,5,8,5,0,06:00-11:59,0,Urban,PASS,PASS,PASS,P

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM trusted_silver_vehicles
        LIMIT 20
    """)
)


collision_index,collision_year,collision_ref_no,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre_historic,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane_historic,vehicle_location_restricted_lane,junction_location,skidding_and_overturning,hit_object_in_carriageway,vehicle_leaving_carriageway,hit_object_off_carriageway,first_point_of_impact,vehicle_left_hand_drive,journey_purpose_of_driver_historic,journey_purpose_of_driver,sex_of_driver,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding,ingested_at,source_file,vehicle_key,dq02_status,dq08_status,overall_status
2024471526976,2024,471526976,2,4,0,-1,19,4,8,-1,0,8,0,0,0,0,3,1,-1,9,1,23,5,234,1,16,MAKE AND MODEL REDACTED,8,E01020954,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471526976_2,PASS,PASS,PASS
2024471493223,2024,471493223,2,9,0,-1,2,0,0,-1,0,0,0,0,0,0,1,1,-1,6,-1,-1,-1,2198,1,15,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471493223_2,PASS,PASS,PASS
2024010504128,2024,010504128,2,9,0,2,2,0,0,0,0,0,0,0,0,0,2,1,6,6,3,-1,-1,2198,1,18,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024010504128_2,PASS,PASS,PASS
2024302400896,2024,302400896,2,9,0,18,19,1,4,0,0,0,0,0,0,0,1,1,6,6,1,56,9,6496,1,0,MAKE AND MODEL REDACTED,5,E01019782,0,2,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024302400896_2,PASS,PASS,PASS
2024631472488,2024,631472488,2,90,0,-1,19,7,3,-1,0,8,0,0,0,0,4,1,-1,9,1,55,8,2184,2,0,MAKE AND MODEL REDACTED,7,W01000623,0,4,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024631472488_2,PASS,PASS,PASS
2024471512096,2024,471512096,1,4,0,-1,19,2,6,-1,0,0,2,0,0,0,4,1,-1,9,1,57,9,449,1,0,MAKE AND MODEL REDACTED,5,E01021074,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471512096_1,PASS,PASS,PASS
2024211495435,2024,211495435,1,2,0,-1,3,3,3,-1,0,1,5,0,0,0,2,1,-1,6,2,17,4,49,1,21,MAKE AND MODEL REDACTED,3,E01029752,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024211495435_1,PASS,PASS,PASS
2024131459645,2024,131459645,1,9,0,-1,10,1,5,-1,0,8,0,0,0,0,3,1,-1,9,1,44,7,1591,1,5,HYUNDAI TUCSON,7,E01011942,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024131459645_1,PASS,PASS,PASS
2024401457067,2024,401457067,1,9,0,-1,6,7,3,-1,0,0,0,0,0,0,1,1,-1,6,1,26,6,1685,2,9,HYUNDAI TUCSON,10,E01017611,0,3,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024401457067_1,PASS,PASS,PASS
2024031497068,2024,031497068,1,9,0,-1,19,1,5,-1,0,0,5,0,1,4,1,1,-1,9,1,48,8,1598,1,3,HYUNDAI TUCSON,1,-1,0,5,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024031497068_1,PASS,PASS,PASS


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM trusted_silver_vehicles
        LIMIT 20
    """)
)

collision_index,collision_year,collision_ref_no,vehicle_reference,vehicle_type,towing_and_articulation,vehicle_manoeuvre_historic,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane_historic,vehicle_location_restricted_lane,junction_location,skidding_and_overturning,hit_object_in_carriageway,vehicle_leaving_carriageway,hit_object_off_carriageway,first_point_of_impact,vehicle_left_hand_drive,journey_purpose_of_driver_historic,journey_purpose_of_driver,sex_of_driver,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding,ingested_at,source_file,vehicle_key,dq02_status,dq08_status,overall_status
2024471526976,2024,471526976,2,4,0,-1,19,4,8,-1,0,8,0,0,0,0,3,1,-1,9,1,23,5,234,1,16,MAKE AND MODEL REDACTED,8,E01020954,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471526976_2,PASS,PASS,PASS
2024471493223,2024,471493223,2,9,0,-1,2,0,0,-1,0,0,0,0,0,0,1,1,-1,6,-1,-1,-1,2198,1,15,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471493223_2,PASS,PASS,PASS
2024010504128,2024,010504128,2,9,0,2,2,0,0,0,0,0,0,0,0,0,2,1,6,6,3,-1,-1,2198,1,18,MAKE AND MODEL REDACTED,-1,-1,0,-1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024010504128_2,PASS,PASS,PASS
2024302400896,2024,302400896,2,9,0,18,19,1,4,0,0,0,0,0,0,0,1,1,6,6,1,56,9,6496,1,0,MAKE AND MODEL REDACTED,5,E01019782,0,2,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024302400896_2,PASS,PASS,PASS
2024631472488,2024,631472488,2,90,0,-1,19,7,3,-1,0,8,0,0,0,0,4,1,-1,9,1,55,8,2184,2,0,MAKE AND MODEL REDACTED,7,W01000623,0,4,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024631472488_2,PASS,PASS,PASS
2024471512096,2024,471512096,1,4,0,-1,19,2,6,-1,0,0,2,0,0,0,4,1,-1,9,1,57,9,449,1,0,MAKE AND MODEL REDACTED,5,E01021074,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024471512096_1,PASS,PASS,PASS
2024211495435,2024,211495435,1,2,0,-1,3,3,3,-1,0,1,5,0,0,0,2,1,-1,6,2,17,4,49,1,21,MAKE AND MODEL REDACTED,3,E01029752,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024211495435_1,PASS,PASS,PASS
2024131459645,2024,131459645,1,9,0,-1,10,1,5,-1,0,8,0,0,0,0,3,1,-1,9,1,44,7,1591,1,5,HYUNDAI TUCSON,7,E01011942,0,1,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024131459645_1,PASS,PASS,PASS
2024401457067,2024,401457067,1,9,0,-1,6,7,3,-1,0,0,0,0,0,0,1,1,-1,6,1,26,6,1685,2,9,HYUNDAI TUCSON,10,E01017611,0,3,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024401457067_1,PASS,PASS,PASS
2024031497068,2024,031497068,1,9,0,-1,19,1,5,-1,0,0,5,0,1,4,1,1,-1,9,1,48,8,1598,1,3,HYUNDAI TUCSON,1,-1,0,5,2026-07-31T10:25:03.626Z,dbfs:/Volumes/road_safety/default/pg-13/vehicles.csv,2024031497068_1,PASS,PASS,PASS


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM trusted_silver_casualties
        LIMIT 20
    """)
)

collision_index,collision_year,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding,ingested_at,source_file,casualty_key,casualty_age_group,vulnerable_road_user_flag,severe_casualty_flag,dq03_status,dq07_status,dq08_status,overall_status
2024991534042,2024,991534042,1,1,3,2,39,7,3,1,3,0,0,-1,0,-1,-1,3,1,0,1,-1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024991534042_1,31-50,1,0,PASS,PASS,PASS,PASS
2024430342145,2024,430342145,2,1,1,2,25,5,3,0,0,0,0,0,1,4,E01028532,-1,0,null,null,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024430342145_1,18-30,0,0,PASS,PASS,PASS,PASS
2024471442093,2024,471442093,2,1,1,1,49,8,3,0,0,0,0,0,1,8,E01031491,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024471442093_1,31-50,0,0,PASS,PASS,PASS,PASS
2024101511084,2024,101511084,1,1,1,2,39,7,3,0,0,0,0,0,9,1,E01008859,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024101511084_1,31-50,0,0,PASS,PASS,PASS,PASS
2024161531062,2024,161531062,1,1,1,1,59,9,3,0,0,0,0,0,1,5,E01012902,3,1,0,1,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024161531062_1,51-70,0,0,PASS,PASS,PASS,PASS
2024421434943,2024,421434943,2,1,1,1,62,9,2,0,0,0,0,0,3,5,E01034081,5,1,1,0,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024421434943_1,51-70,0,1,PASS,PASS,PASS,PASS
2024010533954,2024,010533954,1,1,1,1,44,7,3,0,0,0,0,0,5,5,E01021790,3,1,0,1,3,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024010533954_1,31-50,0,0,PASS,PASS,PASS,PASS
2024131453448,2024,131453448,1,2,2,2,35,6,2,0,0,1,0,0,9,4,E01011447,7,1,1,0,1,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024131453448_2,31-50,0,1,PASS,PASS,PASS,PASS
2024010511909,2024,010511909,2,1,1,1,34,6,3,0,0,0,0,0,3,5,E01032741,3,1,0,1,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024010511909_1,31-50,0,0,PASS,PASS,PASS,PASS
2024070726166,2024,070726166,2,1,1,1,58,9,2,0,0,0,0,0,3,8,E01014131,7,1,1,0,4,2026-07-31T10:25:01.060Z,dbfs:/Volumes/road_safety/default/pg-13/casualties.csv,2024070726166_1,51-70,0,1,PASS,PASS,PASS,PASS


In [0]:
# WEEK 7 - Step 6: Trusted Silver row counts

display(
    spark.sql("""
        SELECT 'trusted_silver_collisions' AS table_name,
               COUNT(*) AS row_count
        FROM trusted_silver_collisions

        UNION ALL

        SELECT 'trusted_silver_vehicles' AS table_name,
               COUNT(*) AS row_count
        FROM trusted_silver_vehicles

        UNION ALL

        SELECT 'trusted_silver_casualties' AS table_name,
               COUNT(*) AS row_count
        FROM trusted_silver_casualties
    """)
)

table_name,row_count
trusted_silver_collisions,99047
trusted_silver_vehicles,181655
trusted_silver_casualties,124817


In [0]:
# WEEK 7 - Step 7: Profile Trusted Silver collision data

display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT collision_index) AS distinct_collisions,
            COUNT(DISTINCT collision_ref_no) AS distinct_collision_refs,
            COUNT(*) - COUNT(collision_index) AS null_collision_index,
            COUNT(*) - COUNT(collision_timestamp) AS null_collision_timestamp,
            COUNT(*) - COUNT(collision_month) AS null_collision_month,
            COUNT(*) - COUNT(urban_rural_group) AS null_urban_rural_group,
            COUNT(*) - COUNT(severe_collision_flag) AS null_severe_flag
        FROM trusted_silver_collisions
    """)
)

total_rows,distinct_collisions,distinct_collision_refs,null_collision_index,null_collision_timestamp,null_collision_month,null_urban_rural_group,null_severe_flag
99047,98947,98949,100,0,0,0,0


In [0]:
# WEEK 7 - Step 8: Check KPI field values

print("Urban/Rural groups:")
display(
    spark.sql("""
        SELECT
            urban_rural_group,
            COUNT(*) AS row_count
        FROM trusted_silver_collisions
        GROUP BY urban_rural_group
        ORDER BY row_count DESC
    """)
)

print("Severe collision flag:")
display(
    spark.sql("""
        SELECT
            severe_collision_flag,
            COUNT(*) AS row_count
        FROM trusted_silver_collisions
        GROUP BY severe_collision_flag
        ORDER BY severe_collision_flag
    """)
)

print("Collision months:")
display(
    spark.sql("""
        SELECT
            collision_month,
            COUNT(*) AS row_count
        FROM trusted_silver_collisions
        GROUP BY collision_month
        ORDER BY collision_month
    """)
)

Urban/Rural groups:


urban_rural_group,row_count
Urban,66031
Rural,33013
Unknown,3


Severe collision flag:


severe_collision_flag,row_count
0,74540
1,24507


Collision months:


collision_month,row_count
1,8160
2,7397
3,7663
4,7506
5,8809
6,8874
7,8365
8,8049
9,8670
10,8902


In [0]:
# WEEK 7 - Step 9: Trusted Silver quality status

display(
    spark.sql("""
        SELECT
            overall_status,
            COUNT(*) AS row_count
        FROM trusted_silver_collisions
        GROUP BY overall_status
        ORDER BY row_count DESC
    """)
)

overall_status,row_count
PASS,99047


In [0]:
# WEEK 7 - Step 9B: DQ status profile

display(
    spark.sql("""
        SELECT
            dq01_status,
            dq04_status,
            dq05_status,
            dq06_status,
            dq08_status,
            overall_status,
            COUNT(*) AS row_count
        FROM trusted_silver_collisions
        GROUP BY
            dq01_status,
            dq04_status,
            dq05_status,
            dq06_status,
            dq08_status,
            overall_status
        ORDER BY row_count DESC
        LIMIT 50
    """)
)

dq01_status,dq04_status,dq05_status,dq06_status,dq08_status,overall_status,row_count
PASS,PASS,PASS,PASS,PASS,PASS,98947
null,null,null,null,null,PASS,100


In [0]:
# WEEK 7 - Step 10: Collision key uniqueness

display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT collision_index) AS distinct_collision_keys,
            COUNT(*) - COUNT(DISTINCT collision_index) AS duplicate_rows
        FROM trusted_silver_collisions
    """)
)

total_rows,distinct_collision_keys,duplicate_rows
99047,98947,100


In [0]:
# WEEK 7 - Step 11: Collision date coverage

display(
    spark.sql("""
        SELECT
            MIN(collision_timestamp) AS first_collision,
            MAX(collision_timestamp) AS last_collision,
            COUNT(DISTINCT collision_year) AS number_of_years,
            COUNT(DISTINCT collision_month) AS number_of_months
        FROM trusted_silver_collisions
    """)
)

first_collision,last_collision,number_of_years,number_of_months
2024-01-01T00:02:00.000Z,2024-12-31T23:30:00.000Z,1,12


In [0]:
# WEEK 7 - Step 12: Identify duplicate collision keys

display(
    spark.sql("""
        SELECT
            collision_index,
            COUNT(*) AS occurrence_count
        FROM trusted_silver_collisions
        GROUP BY collision_index
        HAVING COUNT(*) > 1
        ORDER BY occurrence_count DESC, collision_index
    """)
)

collision_index,occurrence_count
null,100


In [0]:
# WEEK 7 - Step 13: Inspect duplicated collision records

display(
    spark.sql("""
        SELECT *
        FROM trusted_silver_collisions
        WHERE collision_index IN (
            SELECT collision_index
            FROM trusted_silver_collisions
            GROUP BY collision_index
            HAVING COUNT(*) > 1
        )
        ORDER BY collision_index
        LIMIT 100
    """)
)

collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,local_authority_district,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail_historic,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control_historic,pedestrian_crossing_physical_facilities_historic,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,ingested_at,source_file,collision_timestamp,collision_month,collision_hour,collision_weekday,weekend_flag,time_band,severe_collision_flag,urban_rural_group,dq01_status,dq04_status,dq05_status,dq06_status,dq08_status,overall_status


In [0]:
# WEEK 7 - Step 14: Check exact duplicate rows

display(
    spark.sql("""
        SELECT
            collision_index,
            collision_year,
            collision_ref_no,
            collision_timestamp,
            collision_month,
            urban_rural_group,
            severe_collision_flag,
            COUNT(*) AS occurrence_count
        FROM trusted_silver_collisions
        GROUP BY
            collision_index,
            collision_year,
            collision_ref_no,
            collision_timestamp,
            collision_month,
            urban_rural_group,
            severe_collision_flag
        HAVING COUNT(*) > 1
        ORDER BY occurrence_count DESC
    """)
)

collision_index,collision_year,collision_ref_no,collision_timestamp,collision_month,urban_rural_group,severe_collision_flag,occurrence_count


In [0]:
# WEEK 7 - Step 15: Investigate NULL collision keys

display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(collision_index) AS non_null_collision_index,
            COUNT(*) - COUNT(collision_index) AS null_collision_index
        FROM trusted_silver_collisions
    """)
)

total_rows,non_null_collision_index,null_collision_index
99047,98947,100


In [0]:
# WEEK 7 - Step 16: Inspect rows with NULL collision_index

display(
    spark.sql("""
        SELECT *
        FROM trusted_silver_collisions
        WHERE collision_index IS NULL
        LIMIT 100
    """)
)

collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,local_authority_district,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail_historic,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control_historic,pedestrian_crossing_physical_facilities_historic,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,ingested_at,source_file,collision_timestamp,collision_month,collision_hour,collision_weekday,weekend_flag,time_band,severe_collision_flag,urban_rural_group,dq01_status,dq04_status,dq05_status,dq06_status,dq08_status,overall_status
null,2024,622400113,313754,168792,-3.24149,51.41157,62,3,2,2,29/01/2024,2,00:29,-1,W06000014,W06000014,W06000014,3,4055,6,20,3,13,2,6,0,0,0,0,1,1,1,0,0,0,1,1,-1,W01001071,-1,0,null,null,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-01-29T00:29:00.000Z,1,0,2,0,00:00-05:59,0,Urban,null,null,null,null,null,PASS
null,2024,991422397,254172,672313,-4.33523,55.92172,99,1,2,1,19/03/2024,3,08:09,-1,S12000045,S12000045,S12000045,6,0,6,30,-1,13,4,3,809,-1,-1,0,1,1,1,-1,-1,0,1,1,-1,-1,1,1,0,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-03-19T08:09:00.000Z,3,8,3,0,06:00-11:59,1,Urban,null,null,null,null,null,PASS
null,2024,991473707,249850,665729,-4.40067,55.86129,99,2,3,3,29/07/2024,2,12:24,-1,S12000038,S12000038,S12000038,1,8,3,70,-1,0,-1,-1,-1,-1,-1,0,1,1,1,-1,-1,0,1,1,-1,-1,6,1,1,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-07-29T12:24:00.000Z,7,12,2,0,12:00-17:59,1,Urban,null,null,null,null,null,PASS
null,2024,061467980,388904,390863,-2.1684,53.41445,6,3,2,1,15/07/2024,2,11:30,-1,E08000007,E08000007,E08000007,3,6,6,30,-1,0,-1,6,0,-1,-1,0,1,1,1,-1,-1,0,1,1,2,E01005917,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-07-15T11:30:00.000Z,7,11,2,0,06:00-11:59,0,Urban,null,null,null,null,null,PASS
null,2024,041442753,366627,427252,-2.50745,53.74057,4,2,2,1,17/04/2024,4,22:30,-1,E06000008,E06000008,E06000008,3,674,6,30,-1,13,2,4,6447,-1,-1,0,4,2,2,-1,-1,0,1,1,2,E01012623,7,1,1,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-04-17T22:30:00.000Z,4,22,4,0,18:00-23:59,1,Urban,null,null,null,null,null,PASS
null,2024,622400635,317398,172763,-3.19003,51.44781,62,3,1,1,30/08/2024,6,23:10,-1,W06000014,W06000014,W06000014,3,4160,3,30,6,16,2,3,4055,0,5,15,4,1,1,0,0,0,1,1,-1,W01001076,-1,0,null,null,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-08-30T23:10:00.000Z,8,23,6,0,18:00-23:59,0,Urban,null,null,null,null,null,PASS
null,2024,161458437,490322,413135,-0.63638,53.607,16,3,2,1,21/06/2024,6,14:05,-1,E06000013,E06000013,E06000013,3,1077,6,50,-1,0,-1,-1,-1,-1,-1,0,1,1,1,-1,-1,0,2,3,2,E01013297,3,1,0,1,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-06-21T14:05:00.000Z,6,14,6,0,12:00-17:59,0,Rural,null,null,null,null,null,PASS
null,2024,991472789,250808,685328,-4.39622,56.03752,99,2,2,2,27/07/2024,7,12:22,-1,S12000030,S12000030,S12000030,3,81,6,60,-1,19,4,4,834,-1,-1,0,1,1,1,-1,-1,0,2,1,-1,-1,6,1,1,0,2026-07-31T10:24:57.570Z,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2024-07-27T12:22:00.000Z,7,12,7,1,12:00-17:59,1,Rural,null,null,null,null,null,PASS
null,2024,101475454,418064,564902,-1.7193,54.9784,10,2,2,1,02/08/

In [0]:
# WEEK 7 - Step 17: Check alternative collision identifiers

display(
    spark.sql("""
        SELECT
            COUNT(*) AS null_index_rows,
            COUNT(collision_ref_no) AS non_null_ref_no,
            COUNT(DISTINCT collision_ref_no) AS distinct_ref_no
        FROM trusted_silver_collisions
        WHERE collision_index IS NULL
    """)
)


null_index_rows,non_null_ref_no,distinct_ref_no
100,100,100


In [0]:
# WEEK 7 - Step 18: Validate collision_ref_no uniqueness

display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(collision_ref_no) AS non_null_ref_no,
            COUNT(DISTINCT collision_ref_no) AS distinct_ref_no,
            COUNT(*) - COUNT(collision_ref_no) AS null_ref_no
        FROM trusted_silver_collisions
    """)
)

total_rows,non_null_ref_no,distinct_ref_no,null_ref_no
99047,99047,98949,0


In [0]:
# WEEK 7 - Step 18B: Find duplicate collision_ref_no values

display(
    spark.sql("""
        SELECT
            collision_ref_no,
            COUNT(*) AS occurrence_count
        FROM trusted_silver_collisions
        WHERE collision_ref_no IS NOT NULL
        GROUP BY collision_ref_no
        HAVING COUNT(*) > 1
        ORDER BY occurrence_count DESC
    """)
)

collision_ref_no,occurrence_count
451506253,2
471532019,2
531466682,2
371434659,2
451518379,2
010532046,2
010489104,2
531502515,2
041560228,2
430201950,2


In [0]:
# WEEK 7 - Step 19: Quality status of NULL collision_index rows

display(
    spark.sql("""
        SELECT
            overall_status,
            COUNT(*) AS row_count
        FROM trusted_silver_collisions
        WHERE collision_index IS NULL
        GROUP BY overall_status
        ORDER BY row_count DESC
    """)
)

overall_status,row_count
PASS,100


In [0]:
# WEEK 7 - Step 19B: Check important fields for NULL-index rows

display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(collision_ref_no) AS collision_ref_no_present,
            COUNT(collision_timestamp) AS timestamp_present,
            COUNT(collision_month) AS month_present,
            COUNT(collision_year) AS year_present,
            COUNT(urban_rural_group) AS urban_rural_present,
            COUNT(severe_collision_flag) AS severe_flag_present
        FROM trusted_silver_collisions
        WHERE collision_index IS NULL
    """)
)

total_rows,collision_ref_no_present,timestamp_present,month_present,year_present,urban_rural_present,severe_flag_present
100,100,100,100,100,100,100


In [0]:
# WEEK 7 - Step 20: Validate collision_ref_no across the full table

display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(collision_ref_no) AS non_null_ref_no,
            COUNT(DISTINCT collision_ref_no) AS distinct_ref_no,
            COUNT(*) - COUNT(collision_ref_no) AS null_ref_no
        FROM trusted_silver_collisions
    """)
)

total_rows,non_null_ref_no,distinct_ref_no,null_ref_no
99047,99047,98949,0


In [0]:
# WEEK 7 - Step 20B: Find duplicate collision_ref_no values

display(
    spark.sql("""
        SELECT
            collision_ref_no,
            COUNT(*) AS occurrence_count
        FROM trusted_silver_collisions
        WHERE collision_ref_no IS NOT NULL
        GROUP BY collision_ref_no
        HAVING COUNT(*) > 1
        ORDER BY occurrence_count DESC
    """)
)

collision_ref_no,occurrence_count
451506253,2
471532019,2
531466682,2
371434659,2
451518379,2
010532046,2
010489104,2
531502515,2
041560228,2
430201950,2


In [0]:
# WEEK 7 - Step 21: Inspect duplicate collision references

display(
    spark.sql("""
        SELECT
            collision_ref_no,
            collision_index,
            collision_year,
            collision_timestamp,
            collision_month,
            collision_hour,
            urban_rural_group,
            collision_severity,
            number_of_vehicles,
            number_of_casualties,
            severe_collision_flag,
            overall_status,
            source_file,
            ingested_at
        FROM trusted_silver_collisions
        WHERE collision_ref_no IN (
            SELECT collision_ref_no
            FROM trusted_silver_collisions
            GROUP BY collision_ref_no
            HAVING COUNT(*) > 1
        )
        ORDER BY collision_ref_no, ingested_at
    """)
)

collision_ref_no,collision_index,collision_year,collision_timestamp,collision_month,collision_hour,urban_rural_group,collision_severity,number_of_vehicles,number_of_casualties,severe_collision_flag,overall_status,source_file,ingested_at
010489104,null,2024,2024-01-14T14:22:00.000Z,1,14,Urban,3,2,1,0,PASS,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2026-07-31T10:24:57.570Z
010489104,2024010489104,2024,2024-01-14T14:22:00.000Z,1,14,Urban,3,2,1,0,PASS,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2026-07-31T10:24:57.570Z
010491956,2024010491956,2024,2024-01-30T21:57:00.000Z,1,21,Urban,2,1,1,1,PASS,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2026-07-31T10:24:57.570Z
010491956,null,2024,2024-01-30T21:57:00.000Z,1,21,Urban,2,1,1,1,PASS,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2026-07-31T10:24:57.570Z
010504306,null,2024,2024-04-10T18:55:00.000Z,4,18,Urban,3,2,1,0,PASS,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2026-07-31T10:24:57.570Z
010504306,2024010504306,2024,2024-04-10T18:55:00.000Z,4,18,Urban,3,2,1,0,PASS,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2026-07-31T10:24:57.570Z
010506945,null,2024,2024-04-25T18:15:00.000Z,4,18,Urban,3,1,1,0,PASS,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2026-07-31T10:24:57.570Z
010506945,2024010506945,2024,2024-04-25T18:15:00.000Z,4,18,Urban,3,1,1,0,PASS,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2026-07-31T10:24:57.570Z
010510517,2024010510517,2024,2024-05-09T17:50:00.000Z,5,17,Urban,3,2,1,0,PASS,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2026-07-31T10:24:57.570Z
010510517,null,2024,2024-05-09T17:50:00.000Z,5,17,Urban,3,2,1,0,PASS,dbfs:/Volumes/road_safety/default/pg-13/collisions.csv,2026-07-31T10:24:57.570Z


In [0]:
# WEEK 7 - Step 22: Compare duplicate records

display(
    spark.sql("""
        SELECT
            collision_ref_no,
            COUNT(*) AS total_rows,
            COUNT(DISTINCT collision_index) AS different_collision_indexes,
            COUNT(DISTINCT collision_timestamp) AS different_timestamps,
            COUNT(DISTINCT collision_severity) AS different_severities,
            COUNT(DISTINCT number_of_vehicles) AS different_vehicle_counts,
            COUNT(DISTINCT number_of_casualties) AS different_casualty_counts,
            COUNT(DISTINCT severe_collision_flag) AS different_severe_flags,
            COUNT(DISTINCT source_file) AS different_source_files
        FROM trusted_silver_collisions
        GROUP BY collision_ref_no
        HAVING COUNT(*) > 1
        ORDER BY collision_ref_no
    """)
)

collision_ref_no,total_rows,different_collision_indexes,different_timestamps,different_severities,different_vehicle_counts,different_casualty_counts,different_severe_flags,different_source_files
010489104,2,1,1,1,1,1,1,1
010491956,2,1,1,1,1,1,1,1
010504306,2,1,1,1,1,1,1,1
010506945,2,1,1,1,1,1,1,1
010510517,2,1,1,1,1,1,1,1
010511972,2,1,1,1,1,1,1,1
010516282,2,1,1,1,1,1,1,1
010518148,2,1,1,1,1,1,1,1
010519924,2,1,1,1,1,1,1,1
010521441,2,1,1,1,1,1,1,1


In [0]:
# WEEK 7 - Step 21: Validate collision_index as the row-level key

display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(collision_index) AS non_null_collision_index,
            COUNT(DISTINCT collision_index) AS distinct_collision_index,
            COUNT(*) - COUNT(collision_index) AS null_collision_index
        FROM trusted_silver_collisions
    """)
)

total_rows,non_null_collision_index,distinct_collision_index,null_collision_index
99047,98947,98947,100


In [0]:
# WEEK 7 - Step 21B: Find duplicate non-null collision_index values

display(
    spark.sql("""
        SELECT
            collision_index,
            COUNT(*) AS occurrence_count
        FROM trusted_silver_collisions
        WHERE collision_index IS NOT NULL
        GROUP BY collision_index
        HAVING COUNT(*) > 1
        ORDER BY occurrence_count DESC
    """)
)

collision_index,occurrence_count


In [0]:
# WEEK 7 - Step 22: Validate NULL collision_index records

display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_null_index_rows,
            COUNT(collision_ref_no) AS ref_no_present,
            COUNT(collision_timestamp) AS timestamp_present,
            COUNT(collision_year) AS year_present,
            COUNT(collision_month) AS month_present,
            COUNT(urban_rural_group) AS urban_rural_present,
            COUNT(severe_collision_flag) AS severe_flag_present,
            COUNT(overall_status) AS status_present
        FROM trusted_silver_collisions
        WHERE collision_index IS NULL
    """)
)

total_null_index_rows,ref_no_present,timestamp_present,year_present,month_present,urban_rural_present,severe_flag_present,status_present
100,100,100,100,100,100,100,100


In [0]:
# WEEK 7 - Step 23: Profile Gold dimensions

print("Urban/Rural groups")
display(
    spark.sql("""
        SELECT
            urban_rural_group,
            COUNT(*) AS row_count
        FROM trusted_silver_collisions
        GROUP BY urban_rural_group
        ORDER BY row_count DESC
    """)
)

print("Collision years")
display(
    spark.sql("""
        SELECT
            collision_year,
            COUNT(*) AS row_count
        FROM trusted_silver_collisions
        GROUP BY collision_year
        ORDER BY collision_year
    """)
)

print("Collision months")
display(
    spark.sql("""
        SELECT
            collision_month,
            COUNT(*) AS row_count
        FROM trusted_silver_collisions
        GROUP BY collision_month
        ORDER BY collision_month
    """)
)

Urban/Rural groups


urban_rural_group,row_count
Urban,66031
Rural,33013
Unknown,3


Collision years


collision_year,row_count
2024,99047


Collision months


collision_month,row_count
1,8160
2,7397
3,7663
4,7506
5,8809
6,8874
7,8365
8,8049
9,8670
10,8902


In [0]:
# WEEK 7 - Step 24: Validate severe collision flag

display(
    spark.sql("""
        SELECT
            severe_collision_flag,
            COUNT(*) AS row_count
        FROM trusted_silver_collisions
        GROUP BY severe_collision_flag
        ORDER BY severe_collision_flag
    """)
)

severe_collision_flag,row_count
0,74540
1,24507


In [0]:
# WEEK 7 - Step 25: Define eligible collision population

display(
    spark.sql("""
        SELECT
            COUNT(*) AS trusted_input_rows,

            COUNT(
                CASE
                    WHEN collision_index IS NOT NULL
                    THEN 1
                END
            ) AS eligible_rows,

            COUNT(
                CASE
                    WHEN collision_index IS NULL
                    THEN 1
                END
            ) AS outside_scope_rows

        FROM trusted_silver_collisions
    """)
)

trusted_input_rows,eligible_rows,outside_scope_rows
99047,98947,100


In [0]:
# WEEK 7 - Step 26: Eligible population key validation

display(
    spark.sql("""
        SELECT
            COUNT(*) AS eligible_rows,
            COUNT(DISTINCT collision_index) AS distinct_collision_keys,
            COUNT(*) - COUNT(DISTINCT collision_index) AS duplicate_key_rows
        FROM trusted_silver_collisions
        WHERE collision_index IS NOT NULL
    """)
)

eligible_rows,distinct_collision_keys,duplicate_key_rows
98947,98947,0


In [0]:
# WEEK 7 - Step 27: Create Gold collision monthly summary

spark.sql("""
CREATE OR REPLACE TABLE gold_collision_monthly_summary
USING DELTA
AS
SELECT
    collision_year,
    collision_month,
    urban_rural_group,

    COUNT(*) AS collision_count,

    SUM(
        CASE
            WHEN severe_collision_flag = 1 THEN 1
            ELSE 0
        END
    ) AS severe_collision_count,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN severe_collision_flag = 1 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS severe_collision_rate,

    CURRENT_TIMESTAMP() AS gold_created_at

FROM trusted_silver_collisions

WHERE collision_index IS NOT NULL

GROUP BY
    collision_year,
    collision_month,
    urban_rural_group
""")

print("Gold table created successfully.")

Gold table created successfully.


In [0]:
# WEEK 7 - Step 28: Inspect Gold output

display(
    spark.sql("""
        SELECT *
        FROM gold_collision_monthly_summary
        ORDER BY collision_year, collision_month, urban_rural_group
    """)
)

collision_year,collision_month,urban_rural_group,collision_count,severe_collision_count,severe_collision_rate,gold_created_at
2024,1,Rural,2605,682,26.18,2026-09-10T12:54:10.797Z
2024,1,Unknown,1,0,0.00,2026-09-10T12:54:10.797Z
2024,1,Urban,5547,1250,22.53,2026-09-10T12:54:10.797Z
2024,2,Rural,2320,644,27.76,2026-09-10T12:54:10.797Z
2024,2,Urban,5074,1101,21.70,2026-09-10T12:54:10.797Z
2024,3,Rural,2495,689,27.62,2026-09-10T12:54:10.797Z
2024,3,Urban,5162,1131,21.91,2026-09-10T12:54:10.797Z
2024,4,Rural,2465,742,30.10,2026-09-10T12:54:10.797Z
2024,4,Urban,5028,1122,22.32,2026-09-10T12:54:10.797Z
2024,5,Rural,2962,925,31.23,2026-09-10T12:54:10.797Z


In [0]:
# WEEK 7 - Step 29: Gold row count

display(
    spark.sql("""
        SELECT
            COUNT(*) AS gold_rows
        FROM gold_collision_monthly_summary
    """)
)

gold_rows
27


In [0]:
# WEEK 7 - Step 30: Validate Gold grain uniqueness

display(
    spark.sql("""
        SELECT
            collision_year,
            collision_month,
            urban_rural_group,
            COUNT(*) AS row_count
        FROM gold_collision_monthly_summary
        GROUP BY
            collision_year,
            collision_month,
            urban_rural_group
        HAVING COUNT(*) > 1
        ORDER BY row_count DESC
    """)
)

collision_year,collision_month,urban_rural_group,row_count


In [0]:
# WEEK 7 - Step 31: Validate Gold key completeness

display(
    spark.sql("""
        SELECT
            SUM(
                CASE WHEN collision_year IS NULL THEN 1 ELSE 0 END
            ) AS null_year,

            SUM(
                CASE WHEN collision_month IS NULL THEN 1 ELSE 0 END
            ) AS null_month,

            SUM(
                CASE WHEN urban_rural_group IS NULL THEN 1 ELSE 0 END
            ) AS null_urban_rural_group

        FROM gold_collision_monthly_summary
    """)
)

null_year,null_month,null_urban_rural_group
0,0,0


In [0]:
# WEEK 7 - Step 32: Validate Gold measures

display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_gold_rows,

            SUM(
                CASE
                    WHEN collision_count < 0
                    THEN 1 ELSE 0
                END
            ) AS negative_collision_counts,

            SUM(
                CASE
                    WHEN severe_collision_count < 0
                    THEN 1 ELSE 0
                END
            ) AS negative_severe_counts,

            SUM(
                CASE
                    WHEN severe_collision_count > collision_count
                    THEN 1 ELSE 0
                END
            ) AS severe_greater_than_total,

            SUM(
                CASE
                    WHEN severe_collision_rate < 0
                      OR severe_collision_rate > 100
                    THEN 1 ELSE 0
                END
            ) AS invalid_rates

        FROM gold_collision_monthly_summary
    """)
)

total_gold_rows,negative_collision_counts,negative_severe_counts,severe_greater_than_total,invalid_rates
27,0,0,0,0


In [0]:
# WEEK 7 - Step 33: Reconcile collision population

display(
    spark.sql("""
        SELECT
            (
                SELECT COUNT(*)
                FROM trusted_silver_collisions
                WHERE collision_index IS NOT NULL
            ) AS trusted_eligible_rows,

            (
                SELECT SUM(collision_count)
                FROM gold_collision_monthly_summary
            ) AS gold_collision_rows,

            (
                SELECT COUNT(*)
                FROM trusted_silver_collisions
                WHERE collision_index IS NOT NULL
            )
            -
            (
                SELECT SUM(collision_count)
                FROM gold_collision_monthly_summary
            ) AS difference
    """)
)

trusted_eligible_rows,gold_collision_rows,difference
98947,98947,0


In [0]:
# WEEK 7 - Step 34: Reconcile severe collision population

display(
    spark.sql("""
        SELECT
            (
                SELECT COUNT(*)
                FROM trusted_silver_collisions
                WHERE collision_index IS NOT NULL
                  AND severe_collision_flag = 1
            ) AS trusted_severe_rows,

            (
                SELECT SUM(severe_collision_count)
                FROM gold_collision_monthly_summary
            ) AS gold_severe_rows,

            (
                SELECT COUNT(*)
                FROM trusted_silver_collisions
                WHERE collision_index IS NOT NULL
                  AND severe_collision_flag = 1
            )
            -
            (
                SELECT SUM(severe_collision_count)
                FROM gold_collision_monthly_summary
            ) AS difference
    """)
)

trusted_severe_rows,gold_severe_rows,difference
24477,24477,0


In [0]:
# WEEK 7 - Step 35: Scope reconciliation

display(
    spark.sql("""
        SELECT
            COUNT(*) AS trusted_input_rows,

            SUM(
                CASE
                    WHEN collision_index IS NOT NULL
                    THEN 1 ELSE 0
                END
            ) AS eligible_rows,

            SUM(
                CASE
                    WHEN collision_index IS NULL
                    THEN 1 ELSE 0
                END
            ) AS outside_scope_rows,

            SUM(
                CASE
                    WHEN collision_index IS NOT NULL
                    THEN 1 ELSE 0
                END
            )
            +
            SUM(
                CASE
                    WHEN collision_index IS NULL
                    THEN 1 ELSE 0
                END
            ) AS eligible_plus_outside_scope

        FROM trusted_silver_collisions
    """)
)

trusted_input_rows,eligible_rows,outside_scope_rows,eligible_plus_outside_scope
99047,98947,100,99047


In [0]:
# WEEK 7 - Step 36: Confirm Gold Delta table exists

display(
    spark.sql("""
        DESCRIBE DETAIL gold_collision_monthly_summary
    """)
)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,15f8bbe5-77f9-42e9-a9e4-dce27b62192d,workspace.default.gold_collision_monthly_summary,null,,2026-09-10T12:54:09.900Z,2026-09-10T12:54:15.000Z,List(),List(),1,2813,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
# WEEK 7 - Step 37: Capture first Gold business output

spark.sql("""
CREATE OR REPLACE TEMP VIEW gold_run_1 AS
SELECT
    collision_year,
    collision_month,
    urban_rural_group,
    collision_count,
    severe_collision_count,
    severe_collision_rate
FROM gold_collision_monthly_summary
""")

print("First Gold run captured.")

First Gold run captured.


In [0]:
# WEEK 7 - Step 37B: Controlled rerun

spark.sql("""
CREATE OR REPLACE TABLE gold_collision_monthly_summary
USING DELTA
AS
SELECT
    collision_year,
    collision_month,
    urban_rural_group,

    COUNT(*) AS collision_count,

    SUM(
        CASE
            WHEN severe_collision_flag = 1 THEN 1
            ELSE 0
        END
    ) AS severe_collision_count,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN severe_collision_flag = 1 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS severe_collision_rate,

    CURRENT_TIMESTAMP() AS gold_created_at

FROM trusted_silver_collisions

WHERE collision_index IS NOT NULL

GROUP BY
    collision_year,
    collision_month,
    urban_rural_group
""")

print("Second Gold run completed.")

Second Gold run completed.


In [0]:
# WEEK 7 - Step 37B: Controlled rerun

spark.sql("""
CREATE OR REPLACE TABLE gold_collision_monthly_summary
USING DELTA
AS
SELECT
    collision_year,
    collision_month,
    urban_rural_group,

    COUNT(*) AS collision_count,

    SUM(
        CASE
            WHEN severe_collision_flag = 1 THEN 1
            ELSE 0
        END
    ) AS severe_collision_count,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN severe_collision_flag = 1 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS severe_collision_rate,

    CURRENT_TIMESTAMP() AS gold_created_at

FROM trusted_silver_collisions

WHERE collision_index IS NOT NULL

GROUP BY
    collision_year,
    collision_month,
    urban_rural_group
""")

print("Second Gold run completed.")

Second Gold run completed.


In [0]:
# WEEK 7 - Step 37C: Two-way rerun comparison

display(
    spark.sql("""
        SELECT
            'run_1_minus_run_2' AS comparison,
            COUNT(*) AS difference_rows
        FROM gold_run_1 a
        LEFT ANTI JOIN gold_collision_monthly_summary b
        ON  a.collision_year = b.collision_year
        AND a.collision_month = b.collision_month
        AND a.urban_rural_group = b.urban_rural_group
        AND a.collision_count = b.collision_count
        AND a.severe_collision_count = b.severe_collision_count
        AND a.severe_collision_rate = b.severe_collision_rate

        UNION ALL

        SELECT
            'run_2_minus_run_1' AS comparison,
            COUNT(*) AS difference_rows
        FROM gold_collision_monthly_summary b
        LEFT ANTI JOIN gold_run_1 a
        ON  a.collision_year = b.collision_year
        AND a.collision_month = b.collision_month
        AND a.urban_rural_group = b.urban_rural_group
        AND a.collision_count = b.collision_count
        AND a.severe_collision_count = b.severe_collision_count
        AND a.severe_collision_rate = b.severe_collision_rate
    """)
)

comparison,difference_rows
run_1_minus_run_2,0
run_2_minus_run_1,0


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_vehicle_type_summary
USING DELTA
AS
SELECT
    collision_year,
    vehicle_type,
    COUNT(*) AS vehicle_involvement_count,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM trusted_silver_vehicles
WHERE vehicle_key IS NOT NULL
GROUP BY
    collision_year,
    vehicle_type
""")

print("Gold vehicle table created successfully.")

Gold vehicle table created successfully.


In [0]:
display(spark.sql("""
SELECT *
FROM gold_vehicle_type_summary
ORDER BY collision_year, vehicle_type
"""))

collision_year,vehicle_type,vehicle_involvement_count,gold_created_at
2024,-1,1348,2026-09-10T13:12:56.526Z
2024,1,15166,2026-09-10T13:12:56.526Z
2024,2,848,2026-09-10T13:12:56.526Z
2024,3,8759,2026-09-10T13:12:56.526Z
2024,4,1866,2026-09-10T13:12:56.526Z
2024,5,4460,2026-09-10T13:12:56.526Z
2024,8,2888,2026-09-10T13:12:56.526Z
2024,9,124247,2026-09-10T13:12:56.526Z
2024,10,299,2026-09-10T13:12:56.526Z
2024,11,3105,2026-09-10T13:12:56.526Z


In [0]:
display(spark.sql("""
SELECT COUNT(*) AS gold_rows
FROM gold_vehicle_type_summary
"""))

gold_rows
23


In [0]:
display(spark.sql("""
SELECT COUNT(*) AS gold_rows
FROM gold_vehicle_type_summary
"""))

gold_rows
23


In [0]:
display(spark.sql("""
SELECT
    collision_year,
    vehicle_type,
    COUNT(*) AS row_count
FROM gold_vehicle_type_summary
GROUP BY collision_year, vehicle_type
HAVING COUNT(*) > 1
ORDER BY row_count DESC
"""))

collision_year,vehicle_type,row_count


In [0]:
display(spark.sql("""
SELECT
    SUM(CASE WHEN collision_year IS NULL THEN 1 ELSE 0 END) AS null_year,
    SUM(CASE WHEN vehicle_type IS NULL THEN 1 ELSE 0 END) AS null_vehicle_type
FROM gold_vehicle_type_summary
"""))

null_year,null_vehicle_type
0,0


In [0]:
display(spark.sql("""
SELECT
    COUNT(*) AS total_gold_rows,
    SUM(CASE WHEN vehicle_involvement_count < 0 THEN 1 ELSE 0 END)
        AS negative_vehicle_counts
FROM gold_vehicle_type_summary
"""))

total_gold_rows,negative_vehicle_counts
23,0


In [0]:
display(spark.sql("""
SELECT
    (SELECT COUNT(*)
     FROM trusted_silver_vehicles
     WHERE vehicle_key IS NOT NULL) AS trusted_eligible_rows,

    (SELECT SUM(vehicle_involvement_count)
     FROM gold_vehicle_type_summary) AS gold_vehicle_rows,

    (SELECT COUNT(*)
     FROM trusted_silver_vehicles
     WHERE vehicle_key IS NOT NULL)
    -
    (SELECT SUM(vehicle_involvement_count)
     FROM gold_vehicle_type_summary) AS difference
"""))

trusted_eligible_rows,gold_vehicle_rows,difference
181655,181655,0


In [0]:
display(spark.sql("""
SELECT
    COUNT(*) AS trusted_input_rows,

    SUM(CASE WHEN vehicle_key IS NOT NULL THEN 1 ELSE 0 END)
        AS eligible_rows,

    SUM(CASE WHEN vehicle_key IS NULL THEN 1 ELSE 0 END)
        AS outside_scope_rows,

    SUM(CASE WHEN vehicle_key IS NOT NULL THEN 1 ELSE 0 END)
    +
    SUM(CASE WHEN vehicle_key IS NULL THEN 1 ELSE 0 END)
        AS eligible_plus_outside_scope
FROM trusted_silver_vehicles
"""))

trusted_input_rows,eligible_rows,outside_scope_rows,eligible_plus_outside_scope
181655,181655,0,181655


In [0]:
display(spark.sql("""
DESCRIBE DETAIL gold_vehicle_type_summary
"""))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,394f6269-f5cf-41c0-a3b6-c07f7435d852,workspace.default.gold_vehicle_type_summary,null,,2026-09-10T13:12:56.010Z,2026-09-10T13:12:59.000Z,List(),List(),1,1789,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
display(spark.sql("""
DESCRIBE DETAIL gold_vehicle_type_summary
"""))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,394f6269-f5cf-41c0-a3b6-c07f7435d852,workspace.default.gold_vehicle_type_summary,null,,2026-09-10T13:12:56.010Z,2026-09-10T13:12:59.000Z,List(),List(),1,1789,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW gold_vehicle_run_1 AS
SELECT
    collision_year,
    vehicle_type,
    vehicle_involvement_count
FROM gold_vehicle_type_summary
""")

print("First vehicle Gold run captured.")

First vehicle Gold run captured.


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_vehicle_type_summary
USING DELTA
AS
SELECT
    collision_year,
    vehicle_type,
    COUNT(*) AS vehicle_involvement_count,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM trusted_silver_vehicles
WHERE vehicle_key IS NOT NULL
GROUP BY
    collision_year,
    vehicle_type
""")

print("Second vehicle Gold run completed.")

Second vehicle Gold run completed.


In [0]:
display(spark.sql("""
SELECT
    'run_1_minus_run_2' AS comparison,
    COUNT(*) AS difference_rows
FROM gold_vehicle_run_1 a
LEFT ANTI JOIN gold_vehicle_type_summary b
ON a.collision_year = b.collision_year
AND a.vehicle_type = b.vehicle_type
AND a.vehicle_involvement_count = b.vehicle_involvement_count

UNION ALL

SELECT
    'run_2_minus_run_1' AS comparison,
    COUNT(*) AS difference_rows
FROM gold_vehicle_type_summary b
LEFT ANTI JOIN gold_vehicle_run_1 a
ON a.collision_year = b.collision_year
AND a.vehicle_type = b.vehicle_type
AND a.vehicle_involvement_count = b.vehicle_involvement_count
"""))

comparison,difference_rows
run_1_minus_run_2,0
run_2_minus_run_1,0


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_casualty_severity_summary
USING DELTA
AS
SELECT
    collision_year,
    casualty_age_group,
    casualty_severity,
    COUNT(*) AS casualty_count,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM trusted_silver_casualties
WHERE casualty_key IS NOT NULL
GROUP BY
    collision_year,
    casualty_age_group,
    casualty_severity
""")

print("Gold casualty table created successfully.")

Gold casualty table created successfully.


In [0]:
display(spark.sql("""
SELECT *
FROM gold_casualty_severity_summary
ORDER BY collision_year, casualty_age_group, casualty_severity
"""))

collision_year,casualty_age_group,casualty_severity,casualty_count,gold_created_at
2024,18-30,1,353,2026-09-10T13:20:32.479Z
2024,18-30,2,6722,2026-09-10T13:20:32.479Z
2024,18-30,3,27518,2026-09-10T13:20:32.479Z
2024,18-30,8,134,2026-09-10T13:20:32.479Z
2024,31-50,1,389,2026-09-10T13:20:32.479Z
2024,31-50,2,6972,2026-09-10T13:20:32.479Z
2024,31-50,3,31583,2026-09-10T13:20:32.479Z
2024,31-50,8,152,2026-09-10T13:20:32.479Z
2024,51-70,1,403,2026-09-10T13:20:32.479Z
2024,51-70,2,5598,2026-09-10T13:20:32.479Z


In [0]:
display(spark.sql("""
SELECT COUNT(*) AS gold_rows
FROM gold_casualty_severity_summary
"""))

gold_rows
20


In [0]:
display(spark.sql("""
SELECT
    collision_year,
    casualty_age_group,
    casualty_severity,
    COUNT(*) AS row_count
FROM gold_casualty_severity_summary
GROUP BY
    collision_year,
    casualty_age_group,
    casualty_severity
HAVING COUNT(*) > 1
ORDER BY row_count DESC
"""))

collision_year,casualty_age_group,casualty_severity,row_count


In [0]:
display(spark.sql("""
SELECT
    SUM(CASE WHEN collision_year IS NULL THEN 1 ELSE 0 END) AS null_year,
    SUM(CASE WHEN casualty_age_group IS NULL THEN 1 ELSE 0 END) AS null_age_group,
    SUM(CASE WHEN casualty_severity IS NULL THEN 1 ELSE 0 END) AS null_severity
FROM gold_casualty_severity_summary
"""))

null_year,null_age_group,null_severity
0,0,0


In [0]:
display(spark.sql("""
SELECT
    COUNT(*) AS total_gold_rows,
    SUM(
        CASE
            WHEN casualty_count < 0 THEN 1
            ELSE 0
        END
    ) AS negative_casualty_counts
FROM gold_casualty_severity_summary
"""))

total_gold_rows,negative_casualty_counts
20,0


In [0]:
display(spark.sql("""
SELECT
    (SELECT COUNT(*)
     FROM trusted_silver_casualties
     WHERE casualty_key IS NOT NULL) AS trusted_eligible_rows,

    (SELECT SUM(casualty_count)
     FROM gold_casualty_severity_summary) AS gold_casualty_rows,

    (SELECT COUNT(*)
     FROM trusted_silver_casualties
     WHERE casualty_key IS NOT NULL)
    -
    (SELECT SUM(casualty_count)
     FROM gold_casualty_severity_summary) AS difference
"""))

trusted_eligible_rows,gold_casualty_rows,difference
124817,124817,0


In [0]:
display(spark.sql("""
SELECT
    COUNT(*) AS trusted_input_rows,

    SUM(
        CASE WHEN casualty_key IS NOT NULL THEN 1 ELSE 0 END
    ) AS eligible_rows,

    SUM(
        CASE WHEN casualty_key IS NULL THEN 1 ELSE 0 END
    ) AS outside_scope_rows,

    SUM(
        CASE WHEN casualty_key IS NOT NULL THEN 1 ELSE 0 END
    )
    +
    SUM(
        CASE WHEN casualty_key IS NULL THEN 1 ELSE 0 END
    ) AS eligible_plus_outside_scope
FROM trusted_silver_casualties
"""))

trusted_input_rows,eligible_rows,outside_scope_rows,eligible_plus_outside_scope
124817,124817,0,124817


In [0]:
display(spark.sql("""
DESCRIBE DETAIL gold_casualty_severity_summary
"""))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,b57e3bab-efe7-4664-a360-fbed8cb5f3d8,workspace.default.gold_casualty_severity_summary,null,,2026-09-10T13:20:32.062Z,2026-09-10T13:20:34.000Z,List(),List(),1,2086,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW gold_casualty_run_1 AS
SELECT
    collision_year,
    casualty_age_group,
    casualty_severity,
    casualty_count
FROM gold_casualty_severity_summary
""")

print("First casualty Gold run captured.")

First casualty Gold run captured.


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_casualty_severity_summary
USING DELTA
AS
SELECT
    collision_year,
    casualty_age_group,
    casualty_severity,
    COUNT(*) AS casualty_count,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM trusted_silver_casualties
WHERE casualty_key IS NOT NULL
GROUP BY
    collision_year,
    casualty_age_group,
    casualty_severity
""")

print("Second casualty Gold run completed.")

Second casualty Gold run completed.


In [0]:
display(spark.sql("""
SELECT
    'run_1_minus_run_2' AS comparison,
    COUNT(*) AS difference_rows
FROM gold_casualty_run_1 a
LEFT ANTI JOIN gold_casualty_severity_summary b
ON a.collision_year = b.collision_year
AND a.casualty_age_group = b.casualty_age_group
AND a.casualty_severity = b.casualty_severity
AND a.casualty_count = b.casualty_count

UNION ALL

SELECT
    'run_2_minus_run_1' AS comparison,
    COUNT(*) AS difference_rows
FROM gold_casualty_severity_summary b
LEFT ANTI JOIN gold_casualty_run_1 a
ON a.collision_year = b.collision_year
AND a.casualty_age_group = b.casualty_age_group
AND a.casualty_severity = b.casualty_severity
AND a.casualty_count = b.casualty_count
"""))

comparison,difference_rows
run_1_minus_run_2,0
run_2_minus_run_1,0
